# Process Outline
1. to hex table add bridges used for each dest type from bridge table
2. to hex table add coords of destinations from path table
3. to hex table add coords of destinations from destination table 
4. to destination add bridge ids from bride table hexes used columns for destination type
5. to bridge table add destination coords from destination table

In [2]:
import os
import json
import pandas as pd
import geopandas as gpd
pd.set_option('display.max_columns', None)
import numpy as np
from shapely.geometry import LineString, Point, Polygon
import h3
from collections import defaultdict
import ast
import gc

input_folder_path = "/Volumes/samsung-4tb/b2p/impact-model/app-data/"
output_folder_path ="/Volumes/samsung-4tb/b2p/impact-model/database_files/"

In [66]:
# print all files in input folder
files = os.listdir(input_folder_path)
print("Files in input folder:")
for file in files:
    print(file)

Files in input folder:
all_bridges.geojson
all_bridges_tiny.geojson
all_countries_merged_hex8.parquet
all_countries_merged_hex8.geojson
all_bridges_full.parquet
all_urban_centers.geojson
all_bridges.parquet
all_education_facilities.geojson
all_education_paths_geodataframe.parquet
all_health_facilities.geojson
all_health_paths_geodataframe.parquet
all_major_roads.geojson
all_market_paths_geodataframe.parquet
all_education_facilities.parquet
all_health_facilities.parquet
all_major_roads.parquet
all_urban_centers.parquet


In [44]:
bridges_path = os.path.join(input_folder_path, "all_bridges.parquet")
hex_path = os.path.join(input_folder_path, "all_countries_merged_hex8.parquet")

edu_destinations_path = os.path.join(input_folder_path, "all_education_facilities.parquet")
health_destinations_path = os.path.join(input_folder_path, "all_health_facilities.parquet")
markets_destinations_path = os.path.join(input_folder_path, "all_urban_centers.parquet")
roads_destinations_path = os.path.join(input_folder_path, "all_major_roads.parquet")

edu_paths_path = os.path.join(input_folder_path, "all_education_paths_geodataframe.parquet")
health_paths_path = os.path.join(input_folder_path, "all_health_paths_geodataframe.parquet")
markets_paths_path = os.path.join(input_folder_path, "all_market_paths_geodataframe.parquet")


### Cleaning and Fixing Bridge dataframe
The bridge table that I used before didn't have education data in it for some reason. reading in the correct file here. 

In [48]:
bridges = gpd.read_parquet(bridges_path)
bridges

In [49]:
# print bridges columns
print("Bridges columns:")
for col in bridges.columns:
    print(col)

Bridges columns:
bridge_index
type
geometry
subregion_indices
exit_point_index
used_by_cells_for_semi_dense_urban_optimal
used_by_h3_for_semi_dense_urban_optimal
used_by_cells_for_health_posts_optimal
used_by_h3_for_health_posts_optimal
used_by_cells_for_primary_schools_fixed
used_by_h3_for_primary_schools_fixed
used_by_cells_for_all_health_facilities_optimal
used_by_h3_for_all_health_facilities_optimal
used_by_cells_for_health_centers_optimal
used_by_h3_for_health_centers_optimal
used_by_cells_for_major_hospitals_optimal
used_by_h3_for_major_hospitals_optimal
used_by_cells_for_major_roads_optimal
used_by_h3_for_major_roads_optimal
used_by_cells_for_secondary_schools_fixed
used_by_h3_for_secondary_schools_fixed
used_by_cells_for_all_education_facilities_fixed
used_by_h3_for_all_education_facilities_fixed


In [50]:
# drop any columns that start with used_by_cells
bridges = bridges.loc[:, ~bridges.columns.str.startswith('used_by_cells')]
# drop columns subregion_indeces, exit_point_index
bridges = bridges.drop(columns=['subregion_indeces', 'exit_point_index'], errors='ignore')

In [53]:
bridges

,bridge_index,type,geometry,subregion_indices,used_by_h3_for_semi_dense_urban_optimal,used_by_h3_for_health_posts_optimal,used_by_h3_for_primary_schools_fixed,used_by_h3_for_all_health_facilities_optimal,used_by_h3_for_health_centers_optimal,used_by_h3_for_major_hospitals_optimal,used_by_h3_for_major_roads_optimal,used_by_h3_for_secondary_schools_fixed,used_by_h3_for_all_education_facilities_fixed
0,302351,bridge_predicted,POINT (37.46229 4.84980),"[269620, 269629]","[886a4b769bfffff, 886a4b769bfffff, 886a4b7691f...","[886a4b769bfffff, 886a4b769bfffff, 886a4b7691f...",[],"[886a4b39a7fffff, 886a4b39a7fffff, 886a4b39a7f...","[886a4b769bfffff, 886a4b769bfffff, 886a4b7691f...",[],[],[],[]
1,302352,bridge_predicted,POINT (37.46595 4.85105),"[269620, 270228]","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...",[],"[886a4b39a1fffff, 886a4b39a1fffff, 886a4b39a1f...","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...",[],[],[],[]
2,302353,bridge_predicted,POINT (37.46508 4.85778),"[269906, 270228]","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39a9f...","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...",[],[],[],[]
3,302354,bridge_predicted,POINT (37.45727 4.86828),"[267568, 269906]","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...",[],"[886a4b39e7fffff, 886a4b39a9fffff]","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...",[],"[886a4b39e7fffff, 886a4b39a9fffff, 886a4b39a9f...",[],[]
4,302355,bridge_predicted,POINT (37.45522 4.87006),"[267568, 269717]","[886a4b3937fffff, 886a4b3937fffff, 886a4b3937f...","[886a4b3937fffff, 886a4b3937fffff, 886a4b3937f...",[],"[886a4b76d9fffff, 886a4b76d9fffff, 886a4b76d9f...","[886a4b3937fffff, 886a4b3937fffff, 886a4b3937f...",[],"[886a4b76d9fffff, 886a4b76d9fffff, 886a4b76dbf...",[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15207,105971,bridge_predicted,POINT (29.63156 -8.49877),"[149739, 150797]","[8896a9a269fffff, 8896a9b197fffff, 8896a9b197f...","[8896a9a269fffff, 8896a9b197fffff, 8896a9b197f...","[8896a9bad7fffff, 8896a9bad7fffff]","[8896a9a269fffff, 8896a9b197fffff, 8896a9b197f...","[8896a9a269fffff, 8896a9b197fffff, 8896a9b197f...","[8896a9a269fffff, 8896a9b197fffff, 8896a9b197f...",[],"[8896a9a269fffff, 8896a9b197fffff, 8896a9b197f...",[8896a9bad7fffff]
15208,105972,bridge_predicted,POINT (29.63656 -8.48563),"[149147, 150797]","[8896a9a345fffff, 8896a9a345fffff, 8896a9a345f...","[8896a9a225fffff, 8896a9a225fffff, 8896a9a225f...","[8896a9a345fffff, 8896a9a345fffff, 8896a9a345f...","[8896a9a345fffff, 8896a9a345fffff, 8896a9a345f...","[8896a9a345fffff, 8896a9a345fffff, 8896a9a345f...","[8896a9a225fffff, 8896a9a225fffff, 8896a9a225f...",[],"[8896a9a225fffff, 8896a9a225fffff, 8896a9a225f...","[8896a9a345fffff, 8896a9a345fffff, 8896a9a345f..."
15209,105973,bridge_predicted,POINT (29.11544 -8.47639),"[135473, 136007]","[8896f4d2d3fffff, 8896f4d2d3fffff, 8896f4d2d3f...","[8896f4d2d3fffff, 8896f4d2d3fffff, 8896f4d2d3f...","[8896f4d287fffff, 8896f4d287fffff, 8896f4d287f...","[8896f4d287fffff, 8896f4d287fffff, 8896f4d287f...","[8896f4d287fffff, 8896f4d287fffff, 8896f4d287f...","[8896f4d287fffff, 8896f4d287fffff, 8896f4d287f...",[],[],"[8896f4d287fffff, 8896f4d287fffff, 8896f4d287f..."
15210,105974,bridge_predicted,POINT (29.75404 -8.41852),"[153714, 154514]","[8896a98cb3fffff, 8896a98cb3fffff, 8896a98cb3f...","[8896a9b91dfffff, 8896a9b91dfffff, 8896a9b91df...","[8896a9b91dfffff, 8896a9b91dfffff, 8896a9b91df...","[8896a9b91dfffff, 8896a9b91dfffff, 8896a9b91df...","[8896a9b90dfffff, 8896a9b96bfffff, 8896a9b96bf...","[8896a98cb3fffff, 8896a98cb3fffff, 8896a98cb3f...",[],"[8896a98cb3fffff, 8896a98cb3fffff, 8896a98cb3f...","[8896a9b91dfffff, 8896a9b91dfffff, 8896a9b91df..."


In [72]:
# Check what type the values actually are
sample_value = bridges['used_by_h3_for_semi_dense_urban_optimal'].iloc[0]
print(f"Type: {type(sample_value)}")
print(f"Value: {sample_value}")
print(f"Repr: {repr(sample_value)}")

Type: <class 'numpy.ndarray'>
Value: ['886a4b769bfffff' '886a4b769bfffff' '886a4b7691fffff' '886a4b2b69fffff'
 '886a4b7693fffff' '886a4b7693fffff' '886a4b2b45fffff' '886a4b2b69fffff'
 '886a4b2b69fffff' '886a4b2b69fffff' '886a4b2b69fffff' '886a4b2b69fffff'
 '886a4b2b69fffff' '886a4b2b69fffff' '886a4b2b45fffff' '886a4b2b45fffff'
 '886a4b2b45fffff' '886a4b2b45fffff' '886a4b2b45fffff' '886a4b2b45fffff'
 '886a4b2b45fffff' '886a4b2b45fffff' '886a4b2b45fffff' '886a4b2b45fffff'
 '886a4b2b45fffff' '886a4b2b69fffff' '886a4b2b45fffff' '886a4b2b69fffff'
 '886a4b2b6bfffff' '886a4b2b47fffff' '886a4b2b69fffff' '886a4b2b6bfffff'
 '886a4b2b6bfffff' '886a4b2b6bfffff' '886a4b2b6bfffff' '886a4b2b6bfffff'
 '886a4b2b6bfffff' '886a4b2b6bfffff' '886a4b2b6bfffff' '886a4b2b6bfffff'
 '886a4b2b6bfffff' '886a4b2b6bfffff' '886a4b2b0dfffff' '886a4b2b6bfffff'
 '886a4b2b6bfffff' '886a4b2b0dfffff' '886a4b2b6bfffff' '886a4b2b6bfffff'
 '886a4b2b0dfffff' '886a4b2b61fffff' '886a4b2b6bfffff' '886a4b2b0dfffff'
 '886a4b2b61ff

In [67]:
# bridges print the data type in [used_by_h3_for_semi_dense_urban_optimal]
print("Bridges data types:")
for col in bridges.columns:
    print(f"{col}: {bridges[col].dtype}")

Bridges data types:
bridge_index: int64
type: object
geometry: geometry
subregion_indices: object
used_by_h3_for_semi_dense_urban_optimal: object
used_by_h3_for_health_posts_optimal: object
used_by_h3_for_primary_schools_fixed: object
used_by_h3_for_all_health_facilities_optimal: object
used_by_h3_for_health_centers_optimal: object
used_by_h3_for_major_hospitals_optimal: object
used_by_h3_for_major_roads_optimal: object
used_by_h3_for_secondary_schools_fixed: object
used_by_h3_for_all_education_facilities_fixed: object


In [83]:
bridges.to_parquet(os.path.join(input_folder_path, "all_bridges.parquet"), index=False)

### 1. to hex table add bridges used for each dest type from bridge table

In [ ]:
hexes = gpd.read_parquet(hex_path)

In [52]:
bridges

,bridge_index,type,geometry,subregion_indices,used_by_h3_for_semi_dense_urban_optimal,used_by_h3_for_health_posts_optimal,used_by_h3_for_primary_schools_fixed,used_by_h3_for_all_health_facilities_optimal,used_by_h3_for_health_centers_optimal,used_by_h3_for_major_hospitals_optimal,used_by_h3_for_major_roads_optimal,used_by_h3_for_secondary_schools_fixed,used_by_h3_for_all_education_facilities_fixed
0,302351,bridge_predicted,POINT (37.46229 4.84980),"[269620, 269629]","[886a4b769bfffff, 886a4b769bfffff, 886a4b7691f...","[886a4b769bfffff, 886a4b769bfffff, 886a4b7691f...",[],"[886a4b39a7fffff, 886a4b39a7fffff, 886a4b39a7f...","[886a4b769bfffff, 886a4b769bfffff, 886a4b7691f...",[],[],[],[]
1,302352,bridge_predicted,POINT (37.46595 4.85105),"[269620, 270228]","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...",[],"[886a4b39a1fffff, 886a4b39a1fffff, 886a4b39a1f...","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...",[],[],[],[]
2,302353,bridge_predicted,POINT (37.46508 4.85778),"[269906, 270228]","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39a9f...","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...",[],[],[],[]
3,302354,bridge_predicted,POINT (37.45727 4.86828),"[267568, 269906]","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...",[],"[886a4b39e7fffff, 886a4b39a9fffff]","[886a4b2b65fffff, 886a4b2b65fffff, 886a4b2b65f...",[],"[886a4b39e7fffff, 886a4b39a9fffff, 886a4b39a9f...",[],[]
4,302355,bridge_predicted,POINT (37.45522 4.87006),"[267568, 269717]","[886a4b3937fffff, 886a4b3937fffff, 886a4b3937f...","[886a4b3937fffff, 886a4b3937fffff, 886a4b3937f...",[],"[886a4b76d9fffff, 886a4b76d9fffff, 886a4b76d9f...","[886a4b3937fffff, 886a4b3937fffff, 886a4b3937f...",[],"[886a4b76d9fffff, 886a4b76d9fffff, 886a4b76dbf...",[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
15207,105971,bridge_predicted,POINT (29.63156 -8.49877),"[149739, 150797]","[8896a9a269fffff, 8896a9b197fffff, 8896a9b197f...","[8896a9a269fffff, 8896a9b197fffff, 8896a9b197f...","[8896a9bad7fffff, 8896a9bad7fffff]","[8896a9a269fffff, 8896a9b197fffff, 8896a9b197f...","[8896a9a269fffff, 8896a9b197fffff, 8896a9b197f...","[8896a9a269fffff, 8896a9b197fffff, 8896a9b197f...",[],"[8896a9a269fffff, 8896a9b197fffff, 8896a9b197f...",[8896a9bad7fffff]
15208,105972,bridge_predicted,POINT (29.63656 -8.48563),"[149147, 150797]","[8896a9a345fffff, 8896a9a345fffff, 8896a9a345f...","[8896a9a225fffff, 8896a9a225fffff, 8896a9a225f...","[8896a9a345fffff, 8896a9a345fffff, 8896a9a345f...","[8896a9a345fffff, 8896a9a345fffff, 8896a9a345f...","[8896a9a345fffff, 8896a9a345fffff, 8896a9a345f...","[8896a9a225fffff, 8896a9a225fffff, 8896a9a225f...",[],"[8896a9a225fffff, 8896a9a225fffff, 8896a9a225f...","[8896a9a345fffff, 8896a9a345fffff, 8896a9a345f..."
15209,105973,bridge_predicted,POINT (29.11544 -8.47639),"[135473, 136007]","[8896f4d2d3fffff, 8896f4d2d3fffff, 8896f4d2d3f...","[8896f4d2d3fffff, 8896f4d2d3fffff, 8896f4d2d3f...","[8896f4d287fffff, 8896f4d287fffff, 8896f4d287f...","[8896f4d287fffff, 8896f4d287fffff, 8896f4d287f...","[8896f4d287fffff, 8896f4d287fffff, 8896f4d287f...","[8896f4d287fffff, 8896f4d287fffff, 8896f4d287f...",[],[],"[8896f4d287fffff, 8896f4d287fffff, 8896f4d287f..."
15210,105974,bridge_predicted,POINT (29.75404 -8.41852),"[153714, 154514]","[8896a98cb3fffff, 8896a98cb3fffff, 8896a98cb3f...","[8896a9b91dfffff, 8896a9b91dfffff, 8896a9b91df...","[8896a9b91dfffff, 8896a9b91dfffff, 8896a9b91df...","[8896a9b91dfffff, 8896a9b91dfffff, 8896a9b91df...","[8896a9b90dfffff, 8896a9b96bfffff, 8896a9b96bf...","[8896a98cb3fffff, 8896a98cb3fffff, 8896a98cb3f...",[],"[8896a98cb3fffff, 8896a98cb3fffff, 8896a98cb3f...","[8896a9b91dfffff, 8896a9b91dfffff, 8896a9b91df..."


In [74]:
def remove_duplicates_from_arrays(df, columns_to_fix):
    """Remove duplicates from numpy arrays in specified columns"""
    for col in columns_to_fix:
        print(f"Removing duplicates from column: {col}")
        df[col] = df[col].apply(lambda x: np.unique(x) if isinstance(x, np.ndarray) else x)
    print(f"Removed duplicates from columns: {columns_to_fix}")

# Usage - fix your columns first
remove_duplicates_from_arrays(bridges, ['used_by_h3_for_semi_dense_urban_optimal'])
print ("Duplicates removed from 'used_by_h3_for_semi_dense_urban_optimal'.")
remove_duplicates_from_arrays(bridges, ['used_by_h3_for_health_posts_optimal'])
print ("Duplicates removed from 'used_by_h3_for_health_posts_optimal'.")
remove_duplicates_from_arrays(bridges, ['used_by_h3_for_primary_schools_fixed'])
print ("Duplicates removed from 'used_by_h3_for_primary_schools_fixed'.")
remove_duplicates_from_arrays(bridges, ['used_by_h3_for_all_health_facilities_optimal'])
print ("Duplicates removed from 'used_by_h3_for_all_health_facilities_optimal'.")
remove_duplicates_from_arrays(bridges, ['used_by_h3_for_health_centers_optimal'])
print ("Duplicates removed from 'used_by_h3_for_health_centers_optimal'.")
remove_duplicates_from_arrays(bridges, ['used_by_h3_for_major_hospitals_optimal'])
print ("Duplicates removed from 'used_by_h3_for_major_hospitals_optimal'.")
remove_duplicates_from_arrays(bridges, ['used_by_h3_for_major_roads_optimal'])
print ("Duplicates removed from 'used_by_h3_for_major_roads_optimal'.")
remove_duplicates_from_arrays(bridges, ['used_by_h3_for_secondary_schools_fixed'])
print ("Duplicates removed from 'used_by_h3_for_secondary_schools_fixed'.")
remove_duplicates_from_arrays(bridges, ['used_by_h3_for_all_education_facilities_fixed'])
print ("Duplicates removed from 'used_by_h3_for_all_education_facilities_fixed'.")



Removing duplicates from column: used_by_h3_for_semi_dense_urban_optimal
Removed duplicates from columns: ['used_by_h3_for_semi_dense_urban_optimal']
Duplicates removed from 'used_by_h3_for_semi_dense_urban_optimal'.
Removing duplicates from column: used_by_h3_for_health_posts_optimal
Removed duplicates from columns: ['used_by_h3_for_health_posts_optimal']
Duplicates removed from 'used_by_h3_for_health_posts_optimal'.
Removing duplicates from column: used_by_h3_for_primary_schools_fixed
Removed duplicates from columns: ['used_by_h3_for_primary_schools_fixed']
Duplicates removed from 'used_by_h3_for_primary_schools_fixed'.
Removing duplicates from column: used_by_h3_for_all_health_facilities_optimal
Removed duplicates from columns: ['used_by_h3_for_all_health_facilities_optimal']
Duplicates removed from 'used_by_h3_for_all_health_facilities_optimal'.
Removing duplicates from column: used_by_h3_for_health_centers_optimal
Removed duplicates from columns: ['used_by_h3_for_health_centers_op

In [51]:
# print bridges columns
print("Bridges columns:")
for col in bridges.columns:
    print(col)

Bridges columns:
bridge_index
type
geometry
subregion_indices
used_by_h3_for_semi_dense_urban_optimal
used_by_h3_for_health_posts_optimal
used_by_h3_for_primary_schools_fixed
used_by_h3_for_all_health_facilities_optimal
used_by_h3_for_health_centers_optimal
used_by_h3_for_major_hospitals_optimal
used_by_h3_for_major_roads_optimal
used_by_h3_for_secondary_schools_fixed
used_by_h3_for_all_education_facilities_fixed


In [75]:
def add_bridges_to_hexes(bridge_usage_col, output_col):
    """
    Add a column to hexes dataframe with lists of bridge indices that use each hex.
    Modifies the hexes dataframe in place.
    """
    # Build a dictionary mapping hex_id -> set of bridge_ids (no duplicates)
    hex_to_bridges = defaultdict(set)
    
    total_bridges = len(bridges)
    print(f"Processing {total_bridges} bridges...")
    
    for i, (_, bridge_row) in enumerate(bridges.iterrows(), 1):
        bridge_idx = bridge_row['bridge_index']
        used_by_list = bridge_row[bridge_usage_col]
        
        # Handle numpy arrays (from parquet), lists, or other iterables
        if isinstance(used_by_list, (list, np.ndarray)) or hasattr(used_by_list, '__iter__'):
            for hex_idx in used_by_list:
                hex_to_bridges[hex_idx].add(bridge_idx)
        
        # Progress indicator every 1000 rows
        if i % 1000 == 0:
            print(f"Processed {i}/{total_bridges} bridges ({i/total_bridges*100:.1f}%)")
    
    print(f"Finished processing bridges. Mapping to {len(hexes)} hexes...")
    
    # Convert sets to lists and map to hexes (modifies in place)
    hexes[output_col] = hexes['h3_index'].map(
        lambda x: list(hex_to_bridges.get(x, set()))
    )
    
    print(f"Added column '{output_col}' to hexes dataframe")

In [ ]:
# Usage (no assignment needed since it modifies in place):
add_bridges_to_hexes(
    bridge_usage_col='used_by_h3_for_semi_dense_urban_optimal', 
    output_col='bridges_used_for_semi_dense_urban_optimal'
)

Processing 116274 bridges...
Processed 1000/116274 bridges (0.9%)
Processed 2000/116274 bridges (1.7%)
Processed 3000/116274 bridges (2.6%)
Processed 4000/116274 bridges (3.4%)
Processed 5000/116274 bridges (4.3%)
Processed 6000/116274 bridges (5.2%)
Processed 7000/116274 bridges (6.0%)
Processed 8000/116274 bridges (6.9%)
Processed 9000/116274 bridges (7.7%)
Processed 10000/116274 bridges (8.6%)
Processed 11000/116274 bridges (9.5%)
Processed 12000/116274 bridges (10.3%)
Processed 13000/116274 bridges (11.2%)
Processed 14000/116274 bridges (12.0%)
Processed 15000/116274 bridges (12.9%)
Processed 16000/116274 bridges (13.8%)
Processed 17000/116274 bridges (14.6%)
Processed 18000/116274 bridges (15.5%)
Processed 19000/116274 bridges (16.3%)
Processed 20000/116274 bridges (17.2%)
Processed 21000/116274 bridges (18.1%)
Processed 22000/116274 bridges (18.9%)
Processed 23000/116274 bridges (19.8%)
Processed 24000/116274 bridges (20.6%)
Processed 25000/116274 bridges (21.5%)
Processed 26000/

bridge_index
exit_point_index
used_by_h3_for_semi_dense_urban_optimal
used_by_h3_for_health_posts_optimal
used_by_h3_for_all_health_facilities_optimal
used_by_h3_for_health_centers_optimal
used_by_h3_for_major_hospitals_optimal
used_by_h3_for_major_roads_optimal
geometry

In [ ]:
add_bridges_to_hexes(
    bridge_usage_col='used_by_h3_for_health_posts_optimal', 
    output_col='bridges_used_for_health_posts_optimal'
)

Processing 116274 bridges...
Processed 1000/116274 bridges (0.9%)
Processed 2000/116274 bridges (1.7%)
Processed 3000/116274 bridges (2.6%)
Processed 4000/116274 bridges (3.4%)
Processed 5000/116274 bridges (4.3%)
Processed 6000/116274 bridges (5.2%)
Processed 7000/116274 bridges (6.0%)
Processed 8000/116274 bridges (6.9%)
Processed 9000/116274 bridges (7.7%)
Processed 10000/116274 bridges (8.6%)
Processed 11000/116274 bridges (9.5%)
Processed 12000/116274 bridges (10.3%)
Processed 13000/116274 bridges (11.2%)
Processed 14000/116274 bridges (12.0%)
Processed 15000/116274 bridges (12.9%)
Processed 16000/116274 bridges (13.8%)
Processed 17000/116274 bridges (14.6%)
Processed 18000/116274 bridges (15.5%)
Processed 19000/116274 bridges (16.3%)
Processed 20000/116274 bridges (17.2%)
Processed 21000/116274 bridges (18.1%)
Processed 22000/116274 bridges (18.9%)
Processed 23000/116274 bridges (19.8%)
Processed 24000/116274 bridges (20.6%)
Processed 25000/116274 bridges (21.5%)
Processed 26000/

In [ ]:
add_bridges_to_hexes(
    bridge_usage_col='used_by_h3_for_all_health_facilities_optimal', 
    output_col='bridges_used_for_all_health_facilities_optimal'
)

Processing 116274 bridges...
Processed 1000/116274 bridges (0.9%)
Processed 2000/116274 bridges (1.7%)
Processed 3000/116274 bridges (2.6%)
Processed 4000/116274 bridges (3.4%)
Processed 5000/116274 bridges (4.3%)
Processed 6000/116274 bridges (5.2%)
Processed 7000/116274 bridges (6.0%)
Processed 8000/116274 bridges (6.9%)
Processed 9000/116274 bridges (7.7%)
Processed 10000/116274 bridges (8.6%)
Processed 11000/116274 bridges (9.5%)
Processed 12000/116274 bridges (10.3%)
Processed 13000/116274 bridges (11.2%)
Processed 14000/116274 bridges (12.0%)
Processed 15000/116274 bridges (12.9%)
Processed 16000/116274 bridges (13.8%)
Processed 17000/116274 bridges (14.6%)
Processed 18000/116274 bridges (15.5%)
Processed 19000/116274 bridges (16.3%)
Processed 20000/116274 bridges (17.2%)
Processed 21000/116274 bridges (18.1%)
Processed 22000/116274 bridges (18.9%)
Processed 23000/116274 bridges (19.8%)
Processed 24000/116274 bridges (20.6%)
Processed 25000/116274 bridges (21.5%)
Processed 26000/

In [ ]:
add_bridges_to_hexes(
    bridge_usage_col='used_by_h3_for_health_centers_optimal', 
    output_col='bridges_used_for_health_centers_optimal'
)

Processing 116274 bridges...
Processed 1000/116274 bridges (0.9%)
Processed 2000/116274 bridges (1.7%)
Processed 3000/116274 bridges (2.6%)
Processed 4000/116274 bridges (3.4%)
Processed 5000/116274 bridges (4.3%)
Processed 6000/116274 bridges (5.2%)
Processed 7000/116274 bridges (6.0%)
Processed 8000/116274 bridges (6.9%)
Processed 9000/116274 bridges (7.7%)
Processed 10000/116274 bridges (8.6%)
Processed 11000/116274 bridges (9.5%)
Processed 12000/116274 bridges (10.3%)
Processed 13000/116274 bridges (11.2%)
Processed 14000/116274 bridges (12.0%)
Processed 15000/116274 bridges (12.9%)
Processed 16000/116274 bridges (13.8%)
Processed 17000/116274 bridges (14.6%)
Processed 18000/116274 bridges (15.5%)
Processed 19000/116274 bridges (16.3%)
Processed 20000/116274 bridges (17.2%)
Processed 21000/116274 bridges (18.1%)
Processed 22000/116274 bridges (18.9%)
Processed 23000/116274 bridges (19.8%)
Processed 24000/116274 bridges (20.6%)
Processed 25000/116274 bridges (21.5%)
Processed 26000/

In [ ]:
def fix_quoted_list_columns(df, columns_to_fix):
    """Convert string representations of lists to actual lists"""
    for col in columns_to_fix:
        df[col] = df[col].apply(lambda x: ast.literal_eval(x) if isinstance(x, str) else x)
    print(f"Fixed columns: {columns_to_fix}")

fix_quoted_list_columns(bridges, ['used_by_h3_for_major_hospitals_optimal'])
print ("done with major hospitals")
fix_quoted_list_columns(bridges, ['used_by_h3_for_major_roads_optimal'])

Fixed columns: ['used_by_h3_for_major_hospitals_optimal']
done with major hospitals
Fixed columns: ['used_by_h3_for_major_roads_optimal']


In [ ]:
add_bridges_to_hexes(
    bridge_usage_col='used_by_h3_for_major_hospitals_optimal', 
    output_col='bridges_used_for_major_hospitals_optimal'
)

Processing 116274 bridges...
Processed 1000/116274 bridges (0.9%)
Processed 2000/116274 bridges (1.7%)
Processed 3000/116274 bridges (2.6%)
Processed 4000/116274 bridges (3.4%)
Processed 5000/116274 bridges (4.3%)
Processed 6000/116274 bridges (5.2%)
Processed 7000/116274 bridges (6.0%)
Processed 8000/116274 bridges (6.9%)
Processed 9000/116274 bridges (7.7%)
Processed 10000/116274 bridges (8.6%)
Processed 11000/116274 bridges (9.5%)
Processed 12000/116274 bridges (10.3%)
Processed 13000/116274 bridges (11.2%)
Processed 14000/116274 bridges (12.0%)
Processed 15000/116274 bridges (12.9%)
Processed 16000/116274 bridges (13.8%)
Processed 17000/116274 bridges (14.6%)
Processed 18000/116274 bridges (15.5%)
Processed 19000/116274 bridges (16.3%)
Processed 20000/116274 bridges (17.2%)
Processed 21000/116274 bridges (18.1%)
Processed 22000/116274 bridges (18.9%)
Processed 23000/116274 bridges (19.8%)
Processed 24000/116274 bridges (20.6%)
Processed 25000/116274 bridges (21.5%)
Processed 26000/

In [ ]:
add_bridges_to_hexes(
    bridge_usage_col='used_by_h3_for_major_roads_optimal', 
    output_col='bridges_used_for_major_roads_optimal'
)

Processing 116274 bridges...
Processed 1000/116274 bridges (0.9%)
Processed 2000/116274 bridges (1.7%)
Processed 3000/116274 bridges (2.6%)
Processed 4000/116274 bridges (3.4%)
Processed 5000/116274 bridges (4.3%)
Processed 6000/116274 bridges (5.2%)
Processed 7000/116274 bridges (6.0%)
Processed 8000/116274 bridges (6.9%)
Processed 9000/116274 bridges (7.7%)
Processed 10000/116274 bridges (8.6%)
Processed 11000/116274 bridges (9.5%)
Processed 12000/116274 bridges (10.3%)
Processed 13000/116274 bridges (11.2%)
Processed 14000/116274 bridges (12.0%)
Processed 15000/116274 bridges (12.9%)
Processed 16000/116274 bridges (13.8%)
Processed 17000/116274 bridges (14.6%)
Processed 18000/116274 bridges (15.5%)
Processed 19000/116274 bridges (16.3%)
Processed 20000/116274 bridges (17.2%)
Processed 21000/116274 bridges (18.1%)
Processed 22000/116274 bridges (18.9%)
Processed 23000/116274 bridges (19.8%)
Processed 24000/116274 bridges (20.6%)
Processed 25000/116274 bridges (21.5%)
Processed 26000/

In [ ]:
hexes

,h3_index,population,pop_0_4,females_0_4,males_0_4,pop_5_9,females_5_9,males_5_9,pop_10_14,females_10_14,males_10_14,pop_0_9,females_0_9,males_0_9,pop_15_49,females_15_49,males_15_49,pop_50_64,females_50_64,males_50_64,pop_65_plus,females_65_plus,males_65_plus,births,pregnancies,rwi,underweight,female_educational_attainment_mean,male_educational_attainment_mean,travel_time_no_sites_all_health,time_delta_no_sites_semi_dense_urban,travel_time_health_posts,travel_time_major_roads,travel_time_no_sites_secondary_schools,travel_time_secondary_schools,travel_time_no_sites_health_centers,travel_time_no_sites_major_roads,time_delta_no_sites_secondary_schools,time_delta_no_sites_all_health,travel_time_health_centers,time_delta_no_sites_health_centers,time_delta_no_sites_major_roads,travel_time_semi_dense_urban,time_delta_no_sites_major_hospitals,travel_time_all_health,travel_time_no_sites_primary_schools,travel_time_no_sites_semi_dense_urban,time_delta_no_sites_health_posts,travel_time_no_sites_all_education,travel_time_major_hospitals,travel_time_no_sites_major_hospitals,travel_time_primary_schools,time_delta_no_sites_primary_schools,travel_time_all_education,time_delta_no_sites_all_education,travel_time_no_sites_health_posts,geometry,country_name,bridges_used_for_semi_dense_urban_optimal,bridges_used_for_health_posts_optimal,bridges_used_for_all_health_facilities_optimal,bridges_used_for_health_centers_optimal,bridges_used_for_major_hospitals_optimal,bridges_used_for_major_roads_optimal
0,887512209bfffff,5,1,0,0,0,0,0,0,0,0,1,0,0,2,1,1,0,0,0,0,0,0,0,0,-0.277,0.165,3.0,5.1,174.0,0.0,358,59,0.0,0,174.0,59.0,0.0,0.0,174,0.0,0.0,144,0.0,174,NaN,144.0,0.0,NaN,0,0.0,1607,NaN,1599,NaN,358.0,"POLYGON ((-6.50082 7.36543, -6.50476 7.36330, ...",civ,[],[107568],[],[],[],[]
1,8875ae4635fffff,22,3,2,1,3,1,1,2,1,1,7,3,3,10,4,5,1,0,0,0,0,0,0,0,-0.768,0.156,2.4,4.8,858.0,0.0,341,857,0.0,0,858.0,857.0,0.0,0.0,858,0.0,0.0,880,0.0,858,952.0,880.0,0.0,884.0,0,0.0,952,0.0,884,0.0,341.0,"POLYGON ((-8.16296 6.42187, -8.16688 6.41973, ...",civ,[107775],[],[],[],[],[107778]
2,88753244dbfffff,40,7,3,3,6,3,3,4,2,2,13,6,6,17,9,8,3,1,1,1,0,0,0,0,-0.603,0.144,1.9,3.4,739.0,0.0,271,376,430.0,430,739.0,376.0,0.0,0.0,739,0.0,0.0,399,0.0,739,367.0,399.0,0.0,367.0,1154,1154.0,367,0.0,367,0.0,271.0,"POLYGON ((-5.39022 9.68681, -5.39424 9.68467, ...",civ,"[113616, 113620, 113613, 113621]","[113617, 113620, 113621]","[113617, 113620, 113621]",[113607],"[113620, 113621]","[113610, 113613, 113616, 113620, 113621]"
3,8875ab8c3bfffff,8,1,0,0,1,0,0,0,0,0,2,1,1,4,2,2,0,0,0,0,0,0,0,0,-0.400,0.143,3.3,4.9,0.0,0.0,181,64,0.0,0,0.0,64.0,0.0,0.0,0,0.0,0.0,1047,0.0,0,0.0,1047.0,0.0,1426.0,897,897.0,0,0.0,1426,0.0,181.0,"POLYGON ((-7.33689 5.25591, -7.34074 5.25381, ...",civ,"[106087, 106088, 106096, 106098, 106077, 106079]",[],[],[],"[105947, 105948, 105941, 105943]",[]
4,8875ad3897fffff,25,4,2,2,3,1,1,3,1,1,8,3,4,11,5,5,1,0,0,0,0,0,1,2,-0.139,0.160,2.8,4.6,590.0,0.0,272,24,853.0,853,590.0,24.0,0.0,0.0,590,0.0,0.0,250,0.0,590,956.0,250.0,0.0,559.0,998,998.0,956,0.0,559,0.0,272.0,"POLYGON ((-6.22794 6.03084, -6.23181 6.02875, ...",civ,[],[],[],[],[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2064788,8896315601fffff,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-0.571,0.124,4.1,4.8,861.0,0.0,953,981,998.0,998,861.0,981.0,0.0,0.0,861,0.0,0.0,828,0.0,861,479.0,828.0,0.0,479.0,996,996.0,479,0.0,479,0.0,953.0,"POLYGON ((31.07198 -14.27235, 31.07398 -14.267...",zmb,"[98977, 98978, 98975]","[98944, 98945, 98946, 98947, 98951, 98954, 989...","[98944, 98945, 98946, 98947, 98951, 98954, 989...","[98944, 98945, 98946, 98947, 98951, 98954, 989...","[98944, 98945, 98946, 98947, 98886, 98951, 988...","[98944, 98945, 98946, 98947, 98884, 98886, 989..."
2064789,889606a35dfffff,10,1,

In [76]:
add_bridges_to_hexes(
    bridge_usage_col='used_by_h3_for_primary_schools_fixed', 
    output_col='bridges_used_for_primary_schools_fixed'
)

Processing 116274 bridges...
Processed 1000/116274 bridges (0.9%)
Processed 2000/116274 bridges (1.7%)
Processed 3000/116274 bridges (2.6%)
Processed 4000/116274 bridges (3.4%)
Processed 5000/116274 bridges (4.3%)
Processed 6000/116274 bridges (5.2%)
Processed 7000/116274 bridges (6.0%)
Processed 8000/116274 bridges (6.9%)
Processed 9000/116274 bridges (7.7%)
Processed 10000/116274 bridges (8.6%)
Processed 11000/116274 bridges (9.5%)
Processed 12000/116274 bridges (10.3%)
Processed 13000/116274 bridges (11.2%)
Processed 14000/116274 bridges (12.0%)
Processed 15000/116274 bridges (12.9%)
Processed 16000/116274 bridges (13.8%)
Processed 17000/116274 bridges (14.6%)
Processed 18000/116274 bridges (15.5%)
Processed 19000/116274 bridges (16.3%)
Processed 20000/116274 bridges (17.2%)
Processed 21000/116274 bridges (18.1%)
Processed 22000/116274 bridges (18.9%)
Processed 23000/116274 bridges (19.8%)
Processed 24000/116274 bridges (20.6%)
Processed 25000/116274 bridges (21.5%)
Processed 26000/

In [77]:
add_bridges_to_hexes(
    bridge_usage_col='used_by_h3_for_secondary_schools_fixed', 
    output_col='bridges_used_for_secondary_schools_fixed'
)

Processing 116274 bridges...
Processed 1000/116274 bridges (0.9%)
Processed 2000/116274 bridges (1.7%)
Processed 3000/116274 bridges (2.6%)
Processed 4000/116274 bridges (3.4%)
Processed 5000/116274 bridges (4.3%)
Processed 6000/116274 bridges (5.2%)
Processed 7000/116274 bridges (6.0%)
Processed 8000/116274 bridges (6.9%)
Processed 9000/116274 bridges (7.7%)
Processed 10000/116274 bridges (8.6%)
Processed 11000/116274 bridges (9.5%)
Processed 12000/116274 bridges (10.3%)
Processed 13000/116274 bridges (11.2%)
Processed 14000/116274 bridges (12.0%)
Processed 15000/116274 bridges (12.9%)
Processed 16000/116274 bridges (13.8%)
Processed 17000/116274 bridges (14.6%)
Processed 18000/116274 bridges (15.5%)
Processed 19000/116274 bridges (16.3%)
Processed 20000/116274 bridges (17.2%)
Processed 21000/116274 bridges (18.1%)
Processed 22000/116274 bridges (18.9%)
Processed 23000/116274 bridges (19.8%)
Processed 24000/116274 bridges (20.6%)
Processed 25000/116274 bridges (21.5%)
Processed 26000/

In [78]:
add_bridges_to_hexes(
    bridge_usage_col='used_by_h3_for_all_education_facilities_fixed', 
    output_col='bridges_used_for_all_education_facilities_fixed'
)

Processing 116274 bridges...
Processed 1000/116274 bridges (0.9%)
Processed 2000/116274 bridges (1.7%)
Processed 3000/116274 bridges (2.6%)
Processed 4000/116274 bridges (3.4%)
Processed 5000/116274 bridges (4.3%)
Processed 6000/116274 bridges (5.2%)
Processed 7000/116274 bridges (6.0%)
Processed 8000/116274 bridges (6.9%)
Processed 9000/116274 bridges (7.7%)
Processed 10000/116274 bridges (8.6%)
Processed 11000/116274 bridges (9.5%)
Processed 12000/116274 bridges (10.3%)
Processed 13000/116274 bridges (11.2%)
Processed 14000/116274 bridges (12.0%)
Processed 15000/116274 bridges (12.9%)
Processed 16000/116274 bridges (13.8%)
Processed 17000/116274 bridges (14.6%)
Processed 18000/116274 bridges (15.5%)
Processed 19000/116274 bridges (16.3%)
Processed 20000/116274 bridges (17.2%)
Processed 21000/116274 bridges (18.1%)
Processed 22000/116274 bridges (18.9%)
Processed 23000/116274 bridges (19.8%)
Processed 24000/116274 bridges (20.6%)
Processed 25000/116274 bridges (21.5%)
Processed 26000/

In [79]:
add_bridges_to_hexes(
    bridge_usage_col='used_by_h3_for_semi_dense_urban_optimal', 
    output_col='test'
)

Processing 116274 bridges...
Processed 1000/116274 bridges (0.9%)
Processed 2000/116274 bridges (1.7%)
Processed 3000/116274 bridges (2.6%)
Processed 4000/116274 bridges (3.4%)
Processed 5000/116274 bridges (4.3%)
Processed 6000/116274 bridges (5.2%)
Processed 7000/116274 bridges (6.0%)
Processed 8000/116274 bridges (6.9%)
Processed 9000/116274 bridges (7.7%)
Processed 10000/116274 bridges (8.6%)
Processed 11000/116274 bridges (9.5%)
Processed 12000/116274 bridges (10.3%)
Processed 13000/116274 bridges (11.2%)
Processed 14000/116274 bridges (12.0%)
Processed 15000/116274 bridges (12.9%)
Processed 16000/116274 bridges (13.8%)
Processed 17000/116274 bridges (14.6%)
Processed 18000/116274 bridges (15.5%)
Processed 19000/116274 bridges (16.3%)
Processed 20000/116274 bridges (17.2%)
Processed 21000/116274 bridges (18.1%)
Processed 22000/116274 bridges (18.9%)
Processed 23000/116274 bridges (19.8%)
Processed 24000/116274 bridges (20.6%)
Processed 25000/116274 bridges (21.5%)
Processed 26000/

In [ ]:
hexes

,h3_index,population,pop_0_4,females_0_4,males_0_4,pop_5_9,females_5_9,males_5_9,pop_10_14,females_10_14,males_10_14,pop_0_9,females_0_9,males_0_9,pop_15_49,females_15_49,males_15_49,pop_50_64,females_50_64,males_50_64,pop_65_plus,females_65_plus,males_65_plus,births,pregnancies,rwi,underweight,female_educational_attainment_mean,male_educational_attainment_mean,travel_time_no_sites_all_health,time_delta_no_sites_semi_dense_urban,travel_time_health_posts,travel_time_major_roads,travel_time_no_sites_secondary_schools,travel_time_secondary_schools,travel_time_no_sites_health_centers,travel_time_no_sites_major_roads,time_delta_no_sites_secondary_schools,time_delta_no_sites_all_health,travel_time_health_centers,time_delta_no_sites_health_centers,time_delta_no_sites_major_roads,travel_time_semi_dense_urban,time_delta_no_sites_major_hospitals,travel_time_all_health,travel_time_no_sites_primary_schools,travel_time_no_sites_semi_dense_urban,time_delta_no_sites_health_posts,travel_time_no_sites_all_education,travel_time_major_hospitals,travel_time_no_sites_major_hospitals,travel_time_primary_schools,time_delta_no_sites_primary_schools,travel_time_all_education,time_delta_no_sites_all_education,travel_time_no_sites_health_posts,geometry,country_name,bridges_used_for_semi_dense_urban_optimal,bridges_used_for_health_posts_optimal,bridges_used_for_all_health_facilities_optimal,bridges_used_for_health_centers_optimal,bridges_used_for_major_hospitals_optimal,bridges_used_for_major_roads_optimal,bridges_used_for_primary_schools_fixed,bridges_used_for_secondary_schools_fixed,bridges_used_for_all_education_facilities_fixed,test
0,887512209bfffff,5,1,0,0,0,0,0,0,0,0,1,0,0,2,1,1,0,0,0,0,0,0,0,0,-0.277,0.165,3.0,5.1,174.0,0.0,358,59,0.0,0,174.0,59.0,0.0,0.0,174,0.0,0.0,144,0.0,174,NaN,144.0,0.0,NaN,0,0.0,1607,NaN,1599,NaN,358.0,"POLYGON ((-6.50082 7.36543, -6.50476 7.36330, ...",civ,[],[107568],[],[],[],[],[107343],[],[107343],[]
1,8875ae4635fffff,22,3,2,1,3,1,1,2,1,1,7,3,3,10,4,5,1,0,0,0,0,0,0,0,-0.768,0.156,2.4,4.8,858.0,0.0,341,857,0.0,0,858.0,857.0,0.0,0.0,858,0.0,0.0,880,0.0,858,952.0,880.0,0.0,884.0,0,0.0,952,0.0,884,0.0,341.0,"POLYGON ((-8.16296 6.42187, -8.16688 6.41973, ...",civ,[107775],[],[],[],[],[107778],[106649],[],[107775],[107775]
2,88753244dbfffff,40,7,3,3,6,3,3,4,2,2,13,6,6,17,9,8,3,1,1,1,0,0,0,0,-0.603,0.144,1.9,3.4,739.0,0.0,271,376,430.0,430,739.0,376.0,0.0,0.0,739,0.0,0.0,399,0.0,739,367.0,399.0,0.0,367.0,1154,1154.0,367,0.0,367,0.0,271.0,"POLYGON ((-5.39022 9.68681, -5.39424 9.68467, ...",civ,"[113616, 113620, 113613, 113621]","[113617, 113620, 113621]","[113617, 113620, 113621]",[113607],"[113620, 113621]","[113610, 113613, 113616, 113620, 113621]","[113610, 113613, 113616, 113620, 113621]","[113616, 113620, 113613, 113621]","[113610, 113613, 113616, 113620, 113621]","[113616, 113620, 113613, 113621]"
3,8875ab8c3bfffff,8,1,0,0,1,0,0,0,0,0,2,1,1,4,2,2,0,0,0,0,0,0,0,0,-0.400,0.143,3.3,4.9,0.0,0.0,181,64,0.0,0,0.0,64.0,0.0,0.0,0,0.0,0.0,1047,0.0,0,0.0,1047.0,0.0,1426.0,897,897.0,0,0.0,1426,0.0,181.0,"POLYGON ((-7.33689 5.25591, -7.34074 5.25381, ...",civ,"[106087, 106088, 106096, 106098, 106077, 106079]",[],[],[],"[105947, 105948, 105941, 105943]",[],[],[],"[106087, 106088, 106096, 106098, 106077, 106079]","[106087, 106088, 106096, 106098, 106077, 106079]"
4,8875ad3897fffff,25,4,2,2,3,1,1,3,1,1,8,3,4,11,5,5,1,0,0,0,0,0,1,2,-0.139,0.160,2.8,4.6,590.0,0.0,272,24,853.0,853,590.0,24.0,0.0,0.0,590,0.0,0.0,250,0.0,590,956.0,250.0,0.0,559.0,998,998.0,956,0.0,559,0.0,272.0,"POLYGON ((-6.22794 6.03084, -6.23181 6.02875, ...",civ,[],[],[],[],[],[],[],"[106897, 106898, 106900, 106901, 106873]","[112329, 112325, 112326]",[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2064788,8896315601fffff,2,0,0,0,0,0,0,0,0,0,0

In [84]:
hexes = hexes.drop(columns=['test'], errors='ignore')

In [81]:
# whats the longest list in bridges_used_for_secondary_schools_fixed
max_length = max(hexes['bridges_used_for_all_education_facilities_fixed'].apply(len))
print(f"Longest list in 'bridges_used_for_all_education_facilities_fixed': {max_length}")  

Longest list in 'bridges_used_for_all_education_facilities_fixed': 63


In [ ]:
hexes.to_parquet(os.path.join(output_folder_path, "all_countries_merged_hex8_with_bridges.parquet"), index=False)

### 2. to destination files add h3 indexes from path table to each destination type


In [5]:
hexes = gpd.read_parquet(os.path.join(output_folder_path, "all_countries_merged_hex8_with_bridges.parquet"))
hexes

,h3_index,population,pop_0_4,females_0_4,males_0_4,pop_5_9,females_5_9,males_5_9,pop_10_14,females_10_14,males_10_14,pop_0_9,females_0_9,males_0_9,pop_15_49,females_15_49,males_15_49,pop_50_64,females_50_64,males_50_64,pop_65_plus,females_65_plus,males_65_plus,births,pregnancies,rwi,underweight,female_educational_attainment_mean,male_educational_attainment_mean,travel_time_no_sites_all_health,time_delta_no_sites_semi_dense_urban,travel_time_health_posts,travel_time_major_roads,travel_time_no_sites_secondary_schools,travel_time_secondary_schools,travel_time_no_sites_health_centers,travel_time_no_sites_major_roads,time_delta_no_sites_secondary_schools,time_delta_no_sites_all_health,travel_time_health_centers,time_delta_no_sites_health_centers,time_delta_no_sites_major_roads,travel_time_semi_dense_urban,time_delta_no_sites_major_hospitals,travel_time_all_health,travel_time_no_sites_primary_schools,travel_time_no_sites_semi_dense_urban,time_delta_no_sites_health_posts,travel_time_no_sites_all_education,travel_time_major_hospitals,travel_time_no_sites_major_hospitals,travel_time_primary_schools,time_delta_no_sites_primary_schools,travel_time_all_education,time_delta_no_sites_all_education,travel_time_no_sites_health_posts,geometry,country_name,bridges_used_for_semi_dense_urban_optimal,bridges_used_for_health_posts_optimal,bridges_used_for_all_health_facilities_optimal,bridges_used_for_health_centers_optimal,bridges_used_for_major_hospitals_optimal,bridges_used_for_major_roads_optimal,bridges_used_for_primary_schools_fixed,bridges_used_for_secondary_schools_fixed,bridges_used_for_all_education_facilities_fixed
0,887512209bfffff,5,1,0,0,0,0,0,0,0,0,1,0,0,2,1,1,0,0,0,0,0,0,0,0,-0.277,0.165,3.0,5.1,174.0,0.0,358,59,0.0,0,174.0,59.0,0.0,0.0,174,0.0,0.0,144,0.0,174,NaN,144.0,0.0,NaN,0,0.0,1607,NaN,1599,NaN,358.0,"POLYGON ((-6.50082 7.36543, -6.50476 7.36330, ...",civ,[],[107568],[],[],[],[],[107343],[],[107343]
1,8875ae4635fffff,22,3,2,1,3,1,1,2,1,1,7,3,3,10,4,5,1,0,0,0,0,0,0,0,-0.768,0.156,2.4,4.8,858.0,0.0,341,857,0.0,0,858.0,857.0,0.0,0.0,858,0.0,0.0,880,0.0,858,952.0,880.0,0.0,884.0,0,0.0,952,0.0,884,0.0,341.0,"POLYGON ((-8.16296 6.42187, -8.16688 6.41973, ...",civ,[107775],[],[],[],[],[107778],[106649],[],[107775]
2,88753244dbfffff,40,7,3,3,6,3,3,4,2,2,13,6,6,17,9,8,3,1,1,1,0,0,0,0,-0.603,0.144,1.9,3.4,739.0,0.0,271,376,430.0,430,739.0,376.0,0.0,0.0,739,0.0,0.0,399,0.0,739,367.0,399.0,0.0,367.0,1154,1154.0,367,0.0,367,0.0,271.0,"POLYGON ((-5.39022 9.68681, -5.39424 9.68467, ...",civ,"[113616, 113620, 113613, 113621]","[113617, 113620, 113621]","[113617, 113620, 113621]",[113607],"[113620, 113621]","[113610, 113613, 113616, 113620, 113621]","[113610, 113613, 113616, 113620, 113621]","[113616, 113620, 113613, 113621]","[113610, 113613, 113616, 113620, 113621]"
3,8875ab8c3bfffff,8,1,0,0,1,0,0,0,0,0,2,1,1,4,2,2,0,0,0,0,0,0,0,0,-0.400,0.143,3.3,4.9,0.0,0.0,181,64,0.0,0,0.0,64.0,0.0,0.0,0,0.0,0.0,1047,0.0,0,0.0,1047.0,0.0,1426.0,897,897.0,0,0.0,1426,0.0,181.0,"POLYGON ((-7.33689 5.25591, -7.34074 5.25381, ...",civ,"[106087, 106088, 106096, 106098, 106077, 106079]",[],[],[],"[105947, 105948, 105941, 105943]",[],[],[],"[106087, 106088, 106096, 106098, 106077, 106079]"
4,8875ad3897fffff,25,4,2,2,3,1,1,3,1,1,8,3,4,11,5,5,1,0,0,0,0,0,1,2,-0.139,0.160,2.8,4.6,590.0,0.0,272,24,853.0,853,590.0,24.0,0.0,0.0,590,0.0,0.0,250,0.0,590,956.0,250.0,0.0,559.0,998,998.0,956,0.0,559,0.0,272.0,"POLYGON ((-6.22794 6.03084, -6.23181 6.02875, ...",civ,[],[],[],[],[],[],[],"[106897, 106898, 106900, 106901, 106873]","[112329, 112325, 112326]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2064788,8896315601fffff,2,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,-0.571,0.124,4.1,4.8,861.0,0.0,953,981,998.0,998,861.0,981.0,0.0,0.0,861,0.0,0.0,82

#### Education

In [46]:
edu_paths = gpd.read_parquet(edu_paths_path)
edu_paths

,row,col,subregion,coords,exit_point_used,exit_point_index,destination_index,destination_coords,exit_point_index_path,h3_index,exit_point_used_no_sites,exit_point_path_no_sites,path_to_exit_point_no_sites,geometry
0,2173,6678,383896,"[38.565, 13.088333]","[38.56033, 13.08483]",876063,1437,"[38.46577, 13.17027]","[876063, 1086283, 407312, 566356, 344875, 8751...",8852800001fffff,None,None,None,"LINESTRING (38.56500 13.08833, 38.56417 13.087..."
1,2178,6693,385380,"[38.577747, 13.084141]","[38.57775, 13.08414]",67455,1437,"[38.46577, 13.17027]","[67455, 619859, 511176, 489193, 876681, 508811...",8852800003fffff,None,None,None,"LINESTRING (38.57775 13.08414, 38.57775 13.084..."
2,2176,6673,383896,"[38.560833, 13.085834]","[38.56033, 13.08483]",876063,1437,"[38.46577, 13.17027]","[876063, 1086283, 407312, 566356, 344875, 8751...",8852800005fffff,None,None,None,"LINESTRING (38.56083 13.08583, 38.56033 13.084..."
3,2180,6683,384586,"[38.569168, 13.0825]","[38.56224, 13.0697]",550751,1437,"[38.46577, 13.17027]","[550751, 392145, 566356, 344875, 875178, 51945...",8852800007fffff,None,None,None,"LINESTRING (38.56917 13.08250, 38.56917 13.081..."
4,2162,6683,384185,"[38.569168, 13.0975]","[38.57014, 13.09875]",876680,1437,"[38.46577, 13.17027]","[876680, 886026, 508811, 603173, 877186, 50696...",8852800009fffff,None,None,None,"LINESTRING (38.56917 13.09750, 38.57014 13.098..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3466415,11632,11322,440237,"[38.76278, -10.676222]","[38.74946, -10.67406]",367271,10875,"[38.74636, -10.67675]",[367271],88979edb61fffff,None,None,None,"LINESTRING (38.76278 -10.67622, 38.76176 -10.6..."
3466416,11623,11318,440237,"[38.759167, -10.668333]","[38.75639, -10.66906]",226612,10874,"[38.74636, -10.66944]",[226612],88979edb63fffff,None,None,None,"LINESTRING (38.75917 -10.66833, 38.75833 -10.6..."
3466417,11624,11319,440237,"[38.76, -10.669167]","[38.74946, -10.67406]",367271,10875,"[38.74636, -10.67675]",[367271],88979edb63fffff,None,None,None,"LINESTRING (38.76000 -10.66917, 38.75917 -10.6..."
3466418,11618,11324,440237,"[38.76421, -10.66389]","[38.76421, -10.66389]",710632,10874,"[38.74636, -10.66944]","[710632, 364643]",88979edb67fffff,None,None,None,"LINESTRING (38.76421 -10.66389, 38.75923 -10.6..."


In [47]:
edu_destinations = gpd.read_parquet(edu_destinations_path)
edu_destinations

,id,name,geometry,category,coords,all_education_facilities_index,subregion_index,fid,globalid
0,4527811.0,unity university,POINT (38.80434 9.00017),other,"[38.80434, 9.00017]",1,412081,NaN,None
1,4527812.0,cpu college,POINT (38.80249 9.00068),college,"[38.80249, 9.00068]",2,412081,NaN,None
2,4527813.0,ትምህርት ቤት,POINT (38.52361 9.77877),,"[38.52361, 9.77877]",3,378792,NaN,None
3,4527814.0,walya primary,POINT (37.44922 12.59572),primary,"[37.44922, 12.59572]",4,269325,NaN,None
4,4527815.0,further training institute university adama,POINT (39.28836 8.56323),other,"[39.28836, 8.56323]",5,466166,NaN,None
...,...,...,...,...,...,...,...,...,...
104781,NaN,Chikuse Primary School,POINT (28.67089 -14.50118),Primary School,"[28.67089, -14.50118]",7128,123849,8489.0,35LPD800961_002
104782,NaN,Chimpempe Secondary School,POINT (29.43418 -9.54560),Secondary School,"[29.43418, -9.5456]",7129,144698,8490.0,35LQK672438_001
104783,NaN,Kafubu Basic School,POINT (28.58740 -13.02607),Basic School,"[28.5874, -13.02607]",7130,120543,8502.0,35LPF721594_001
104784,NaN,Namiyanga School,POINT (26.50011 -17.07299),School,"[26.50011, -17.07299]",7131,68657,8506.0,35KMB468123_001


In [53]:
# Create a lookup dictionary from edu_paths
coord_to_h3 = {}
for _, row in edu_paths.iterrows():
    coord_key = str(row['destination_coords'])  # Convert coords to string for matching
    h3_value = row['h3_index']
    
    if coord_key not in coord_to_h3:
        coord_to_h3[coord_key] = []
    coord_to_h3[coord_key].append(h3_value)

In [54]:
# Add h3_indices column to edu_destinations
edu_destinations['h3_indices'] = edu_destinations['coords'].apply(
    lambda coord: coord_to_h3.get(str(coord), [])
)

In [ ]:
print(f"Number of unique destination coords in edu_paths: {edu_paths['destination_coords'].astype(str).nunique()}")
print(f"Number of unique destination coords in destinations: {edu_destinations['coords'].astype(str).nunique()}")
print(f"Destinations with h3_indices: {sum(edu_destinations['h3_indices'].apply(len) > 0)}")
edu_destinations[['coords', 'h3_indices']].head()

Number of unique destination coords: 78673
Destinations with h3_indices: 84983


,coords,h3_indices
0,"[38.80434, 9.00017]","[88529b7901fffff, 88529b790dfffff]"
1,"[38.80249, 9.00068]",[]
2,"[38.52361, 9.77877]","[8852980907fffff, 8852980915fffff, 8852980917f..."
3,"[37.44922, 12.59572]","[885282d011fffff, 885282d017fffff, 885282d019f..."
4,"[39.28836, 8.56323]","[887aca0003fffff, 887aca0007fffff, 887aca0011f..."


In [60]:
# write to parquet
edu_destinations.to_parquet(os.path.join(output_folder_path, "all_education_facilities_with_h3.parquet"), index=False)

#### Health

In [ ]:
health_paths = gpd.read_parquet(health_paths_path)
health_destinations = gpd.read_parquet(health_destinations_path)

In [63]:
# Create a lookup dictionary from health_paths
coord_to_h3 = {}
for _, row in health_paths.iterrows():
    coord_key = str(row['destination_coords'])  # Convert coords to string for matching
    h3_value = row['h3_index']
    
    if coord_key not in coord_to_h3:
        coord_to_h3[coord_key] = []
    coord_to_h3[coord_key].append(h3_value)

In [106]:
health_destinations

,facility_type,geometry,coords,all_health_facilities_index,subregion_index,name,fid,globalid,h3_indices
0,clinic,POINT (38.71860 9.02970),"[38.7186, 9.0297]",1,402987,Addis Ketema Clinic 1,NaN,None,[]
1,clinic,POINT (38.73860 9.02922),"[38.7386, 9.02922]",2,404694,Addis Ketema Clinic 10,NaN,None,[]
2,clinic,POINT (38.73620 9.02868),"[38.7362, 9.02868]",3,403784,Addis Ketema Clinic 11,NaN,None,[]
3,clinic,POINT (38.73470 9.02816),"[38.7347, 9.02816]",4,403784,Addis Ketema Clinic 12,NaN,None,[]
4,clinic,POINT (38.73400 9.02942),"[38.734, 9.02942]",5,403784,Addis Ketema Clinic 13,NaN,None,[]
...,...,...,...,...,...,...,...,...,...
26642,rural health post,POINT (23.58512 -16.68015),"[23.58512, -16.68015]",2601,27133,Sankandi RHP,2604.0,{F2987FF1-FEB8-437F-BE54-386D551B385C},"[8897582027fffff, 8897582113fffff, 889758211bf..."
26643,rural health center,POINT (23.15679 -16.19463),"[23.15679, -16.19463]",2602,21382,Silowana RHC,2605.0,{B2A502DC-2FD2-4BB5-9564-489FC20BFA8D},"[88975b1613fffff, 88975b1659fffff, 88975b165bf..."
26644,rural health center,POINT (23.02583 -17.31588),"[23.02583, -17.31588]",2603,19064,Sinjembela RHC,2606.0,{C80AE0FC-E7A1-4E5A-8AF2-BD191C211B4B},"[889758a42dfffff, 889758a445fffff, 889758a44bf..."
26645,rural health center,POINT (23.50325 -16.60110),"[23.50325, -16.6011]",2604,25914,Sioma RHC,2607.0,{B61042BB-4AC6-4D0E-AC37-C75DBC8CFD92},"[8897582883fffff, 8897582889fffff, 889758288bf..."


In [ ]:
health

In [69]:
# Add h3_indices column to edu_destinations
health_destinations['h3_indices'] = health_destinations['coords'].apply(
    lambda coord: coord_to_h3.get(str(coord), [])
)

In [70]:
print(f"Destinations with h3_indices: {sum(health_destinations['h3_indices'].apply(len) > 0)}")
print(f"Number of unique destination coords in health_paths: {health_paths['destination_coords'].astype(str).nunique()}")
print(f"Number of unique destination coords in destinations: {health_destinations['coords'].astype(str).nunique()}")
health_destinations[['coords', 'h3_indices']].head()

Destinations with h3_indices: 25771
Number of unique destination coords in health_paths: 25641
Number of unique destination coords in destinations: 26513


,coords,h3_indices
0,"[38.7186, 9.0297]",[]
1,"[38.7386, 9.02922]",[]
2,"[38.7362, 9.02868]",[]
3,"[38.7347, 9.02816]",[]
4,"[38.734, 9.02942]",[]


In [72]:
# write to parquet
health_destinations.to_parquet(os.path.join(output_folder_path, "all_health_facilities_with_h3.parquet"), index=False)


#### Markets

In [73]:
markets_paths = gpd.read_parquet(markets_paths_path)
market_destinations = gpd.read_parquet(markets_destinations_path)

In [74]:
# Create a lookup dictionary from markets_paths
coord_to_h3 = {}
for _, row in markets_paths.iterrows():
    coord_key = str(row['destination_coords'])  # Convert coords to string for matching
    h3_value = row['h3_index']
    
    if coord_key not in coord_to_h3:
        coord_to_h3[coord_key] = []
    coord_to_h3[coord_key].append(h3_value)

In [75]:
# Add h3_indices column to edu_destinations
market_destinations['h3_indices'] = market_destinations['coords'].apply(
    lambda coord: coord_to_h3.get(str(coord), [])
)

In [79]:
print(f"Destinations with h3_indices: {sum(market_destinations['h3_indices'].apply(len) > 0)}")
print(f"Number of unique destination coords in market_paths: {markets_paths['destination_coords'].astype(str).nunique()}")
print(f"Number of unique destination coords in destinations: {market_destinations['coords'].astype(str).nunique()}")
market_destinations[['coords', 'h3_indices']].head()

Destinations with h3_indices: 2957
Number of unique destination coords in market_paths: 10646
Number of unique destination coords in destinations: 3503


,coords,h3_indices
0,"[42.05792, 4.17458]","[887ae02001fffff, 887ae02003fffff, 887ae02005f..."
1,"[42.17458, 5.22458]","[887ac41001fffff, 887ac41003fffff, 887ac41009f..."
2,"[41.89125, 5.34125]","[887ac40083fffff, 887ac40091fffff, 887ac40093f..."
3,"[43.55792, 5.94125]","[887af02103fffff, 887af02107fffff, 887af02131f..."
4,"[43.54125, 5.94125]","[887af14183fffff, 887af14185fffff, 887af14187f..."


In [80]:
# write to parquet
market_destinations.to_parquet(os.path.join(output_folder_path, "all_urban_centers_with_h3.parquet"), index=False)


### 3. to hex table add coords of destinations from destination table 

In [ ]:
# adding coords from edu_destinations to hexes dataframe
h3_to_coords = {}
for _, row in edu_destinations.iterrows():
    coords = row["coords"]  
    h3_indices = row["h3_indices"]  

    for h3_idx in h3_indices:
        if h3_idx not in h3_to_coords:
            h3_to_coords[h3_idx] = []
        h3_to_coords[h3_idx].append(coords)

hexes['edu_destinations'] = hexes['h3_index'].map(lambda h3: h3_to_coords.get(h3, []))


In [90]:
# adding coords from health_destinations to hexes dataframe
h3_to_coords = {}
for _, row in health_destinations.iterrows():
    coords = row["coords"]  
    h3_indices = row["h3_indices"]  

    for h3_idx in h3_indices:
        if h3_idx not in h3_to_coords:
            h3_to_coords[h3_idx] = []
        h3_to_coords[h3_idx].append(coords)

hexes['health_destinations'] = hexes['h3_index'].map(lambda h3: h3_to_coords.get(h3, []))


In [91]:
hexes

,h3_index,population,pop_0_4,females_0_4,males_0_4,pop_5_9,females_5_9,males_5_9,pop_10_14,females_10_14,males_10_14,pop_0_9,females_0_9,males_0_9,pop_15_49,females_15_49,males_15_49,pop_50_64,females_50_64,males_50_64,pop_65_plus,females_65_plus,males_65_plus,births,pregnancies,rwi,underweight,female_educational_attainment_mean,male_educational_attainment_mean,travel_time_no_sites_all_health,time_delta_no_sites_semi_dense_urban,travel_time_health_posts,travel_time_major_roads,travel_time_no_sites_secondary_schools,travel_time_secondary_schools,travel_time_no_sites_health_centers,travel_time_no_sites_major_roads,time_delta_no_sites_secondary_schools,time_delta_no_sites_all_health,travel_time_health_centers,time_delta_no_sites_health_centers,time_delta_no_sites_major_roads,travel_time_semi_dense_urban,time_delta_no_sites_major_hospitals,travel_time_all_health,travel_time_no_sites_primary_schools,travel_time_no_sites_semi_dense_urban,time_delta_no_sites_health_posts,travel_time_no_sites_all_education,travel_time_major_hospitals,travel_time_no_sites_major_hospitals,travel_time_primary_schools,time_delta_no_sites_primary_schools,travel_time_all_education,time_delta_no_sites_all_education,travel_time_no_sites_health_posts,geometry,country_name,bridges_used_for_semi_dense_urban_optimal,bridges_used_for_health_posts_optimal,bridges_used_for_all_health_facilities_optimal,bridges_used_for_health_centers_optimal,bridges_used_for_major_hospitals_optimal,bridges_used_for_major_roads_optimal,bridges_used_for_primary_schools_fixed,bridges_used_for_secondary_schools_fixed,bridges_used_for_all_education_facilities_fixed,coords_array_x,coords_array_y,edu_destinations,health_destinations
0,887512209bfffff,5,1,0,0,0,0,0,0,0,0,1,0,0,2,1,1,0,0,0,0,0,0,0,0,-0.277,0.165,3.0,5.1,174.0,0.0,358,59,0.0,0,174.0,59.0,0.0,0.0,174,0.0,0.0,144,0.0,174,NaN,144.0,0.0,NaN,0,0.0,1607,NaN,1599,NaN,358.0,"POLYGON ((-6.50082 7.36543, -6.50476 7.36330, ...",civ,[],[107568],[],[],[],[],[107343],[],[107343],[],NaN,"[[-6.44833, 6.91472]]","[[-6.47955, 7.375]]"
1,8875ae4635fffff,22,3,2,1,3,1,1,2,1,1,7,3,3,10,4,5,1,0,0,0,0,0,0,0,-0.768,0.156,2.4,4.8,858.0,0.0,341,857,0.0,0,858.0,857.0,0.0,0.0,858,0.0,0.0,880,0.0,858,952.0,880.0,0.0,884.0,0,0.0,952,0.0,884,0.0,341.0,"POLYGON ((-8.16296 6.42187, -8.16688 6.41973, ...",civ,[107775],[],[],[],[],[107778],[106649],[],[107775],[],NaN,"[[-8.00614, 6.57902]]","[[-8.11667, 6.44413]]"
2,88753244dbfffff,40,7,3,3,6,3,3,4,2,2,13,6,6,17,9,8,3,1,1,1,0,0,0,0,-0.603,0.144,1.9,3.4,739.0,0.0,271,376,430.0,430,739.0,376.0,0.0,0.0,739,0.0,0.0,399,0.0,739,367.0,399.0,0.0,367.0,1154,1154.0,367,0.0,367,0.0,271.0,"POLYGON ((-5.39022 9.68681, -5.39424 9.68467, ...",civ,"[113616, 113620, 113613, 113621]","[113617, 113620, 113621]","[113617, 113620, 113621]",[113607],"[113620, 113621]","[113610, 113613, 113616, 113620, 113621]","[113610, 113613, 113616, 113620, 113621]","[113616, 113620, 113613, 113621]","[113610, 113613, 113616, 113620, 113621]",[],NaN,"[[-5.37773, 9.58976]]","[[-5.435, 9.633]]"
3,8875ab8c3bfffff,8,1,0,0,1,0,0,0,0,0,2,1,1,4,2,2,0,0,0,0,0,0,0,0,-0.400,0.143,3.3,4.9,0.0,0.0,181,64,0.0,0,0.0,64.0,0.0,0.0,0,0.0,0.0,1047,0.0,0,0.0,1047.0,0.0,1426.0,897,897.0,0,0.0,1426,0.0,181.0,"POLYGON ((-7.33689 5.25591, -7.34074 5.25381, ...",civ,"[106087, 106088, 106096, 106098, 106077, 106079]",[],[],[],"[105947, 105948, 105941, 105943]",[],[],[],"[106087, 106088, 106096, 106098, 106077, 106079]",[],NaN,"[[-7.36109, 5.62703]]","[[-7.31698, 5.27485]]"
4,8875ad3897fffff,25,4,2,2,3,1,1,3,1,1,8,3,4,11,5,5,1,0,0,0,0,0,1,2,-0.139,0.160,2.8,4.6,590.0,0.0,272,24,853.0,853,590.0,24.0,0.0,0.0,590,0.0,0.0,250,0.0,590,956.0,250.0,0.0,559.0,998,998.0,956,0.0,559,0.0,272.0,"POLYGON ((-6.22794 6.03084, -6.23181 6.02875, ...",civ,[],[],[],[],[],[],[],"[106897, 106898, 106900, 106901, 106873]","[112329, 112325, 112326]",[],NaN,"[[-6.07698, 5.96967]]","[[-6.30051, 6.00826]]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...

In [92]:
# adding coords from health_destinations to hexes dataframe
h3_to_coords = {}
for _, row in market_destinations.iterrows():
    coords = row["coords"]  
    h3_indices = row["h3_indices"]  

    for h3_idx in h3_indices:
        if h3_idx not in h3_to_coords:
            h3_to_coords[h3_idx] = []
        h3_to_coords[h3_idx].append(coords)

hexes['market_destinations'] = hexes['h3_index'].map(lambda h3: h3_to_coords.get(h3, []))


In [95]:
# drop coords_array_x	coords_array_y 
hexes = hexes.drop(columns=['coords_array_x', 'coords_array_y'], errors='ignore')


In [96]:
hexes

,h3_index,population,pop_0_4,females_0_4,males_0_4,pop_5_9,females_5_9,males_5_9,pop_10_14,females_10_14,males_10_14,pop_0_9,females_0_9,males_0_9,pop_15_49,females_15_49,males_15_49,pop_50_64,females_50_64,males_50_64,pop_65_plus,females_65_plus,males_65_plus,births,pregnancies,rwi,underweight,female_educational_attainment_mean,male_educational_attainment_mean,travel_time_no_sites_all_health,time_delta_no_sites_semi_dense_urban,travel_time_health_posts,travel_time_major_roads,travel_time_no_sites_secondary_schools,travel_time_secondary_schools,travel_time_no_sites_health_centers,travel_time_no_sites_major_roads,time_delta_no_sites_secondary_schools,time_delta_no_sites_all_health,travel_time_health_centers,time_delta_no_sites_health_centers,time_delta_no_sites_major_roads,travel_time_semi_dense_urban,time_delta_no_sites_major_hospitals,travel_time_all_health,travel_time_no_sites_primary_schools,travel_time_no_sites_semi_dense_urban,time_delta_no_sites_health_posts,travel_time_no_sites_all_education,travel_time_major_hospitals,travel_time_no_sites_major_hospitals,travel_time_primary_schools,time_delta_no_sites_primary_schools,travel_time_all_education,time_delta_no_sites_all_education,travel_time_no_sites_health_posts,geometry,country_name,bridges_used_for_semi_dense_urban_optimal,bridges_used_for_health_posts_optimal,bridges_used_for_all_health_facilities_optimal,bridges_used_for_health_centers_optimal,bridges_used_for_major_hospitals_optimal,bridges_used_for_major_roads_optimal,bridges_used_for_primary_schools_fixed,bridges_used_for_secondary_schools_fixed,bridges_used_for_all_education_facilities_fixed,edu_destinations,health_destinations,market_destinations
0,887512209bfffff,5,1,0,0,0,0,0,0,0,0,1,0,0,2,1,1,0,0,0,0,0,0,0,0,-0.277,0.165,3.0,5.1,174.0,0.0,358,59,0.0,0,174.0,59.0,0.0,0.0,174,0.0,0.0,144,0.0,174,NaN,144.0,0.0,NaN,0,0.0,1607,NaN,1599,NaN,358.0,"POLYGON ((-6.50082 7.36543, -6.50476 7.36330, ...",civ,[],[107568],[],[],[],[],[107343],[],[107343],"[[-6.44833, 6.91472]]","[[-6.47955, 7.375]]","[[-6.49125, 7.37792]]"
1,8875ae4635fffff,22,3,2,1,3,1,1,2,1,1,7,3,3,10,4,5,1,0,0,0,0,0,0,0,-0.768,0.156,2.4,4.8,858.0,0.0,341,857,0.0,0,858.0,857.0,0.0,0.0,858,0.0,0.0,880,0.0,858,952.0,880.0,0.0,884.0,0,0.0,952,0.0,884,0.0,341.0,"POLYGON ((-8.16296 6.42187, -8.16688 6.41973, ...",civ,[107775],[],[],[],[],[107778],[106649],[],[107775],"[[-8.00614, 6.57902]]","[[-8.11667, 6.44413]]",[]
2,88753244dbfffff,40,7,3,3,6,3,3,4,2,2,13,6,6,17,9,8,3,1,1,1,0,0,0,0,-0.603,0.144,1.9,3.4,739.0,0.0,271,376,430.0,430,739.0,376.0,0.0,0.0,739,0.0,0.0,399,0.0,739,367.0,399.0,0.0,367.0,1154,1154.0,367,0.0,367,0.0,271.0,"POLYGON ((-5.39022 9.68681, -5.39424 9.68467, ...",civ,"[113616, 113620, 113613, 113621]","[113617, 113620, 113621]","[113617, 113620, 113621]",[113607],"[113620, 113621]","[113610, 113613, 113616, 113620, 113621]","[113610, 113613, 113616, 113620, 113621]","[113616, 113620, 113613, 113621]","[113610, 113613, 113616, 113620, 113621]","[[-5.37773, 9.58976]]","[[-5.435, 9.633]]",[]
3,8875ab8c3bfffff,8,1,0,0,1,0,0,0,0,0,2,1,1,4,2,2,0,0,0,0,0,0,0,0,-0.400,0.143,3.3,4.9,0.0,0.0,181,64,0.0,0,0.0,64.0,0.0,0.0,0,0.0,0.0,1047,0.0,0,0.0,1047.0,0.0,1426.0,897,897.0,0,0.0,1426,0.0,181.0,"POLYGON ((-7.33689 5.25591, -7.34074 5.25381, ...",civ,"[106087, 106088, 106096, 106098, 106077, 106079]",[],[],[],"[105947, 105948, 105941, 105943]",[],[],[],"[106087, 106088, 106096, 106098, 106077, 106079]","[[-7.36109, 5.62703]]","[[-7.31698, 5.27485]]",[]
4,8875ad3897fffff,25,4,2,2,3,1,1,3,1,1,8,3,4,11,5,5,1,0,0,0,0,0,1,2,-0.139,0.160,2.8,4.6,590.0,0.0,272,24,853.0,853,590.0,24.0,0.0,0.0,590,0.0,0.0,250,0.0,590,956.0,250.0,0.0,559.0,998,998.0,956,0.0,559,0.0,272.0,"POLYGON ((-6.22794 6.03084, -6.23181 6.02875, ...",civ,[],[],[],[],[],[],[],"[106897, 106898, 106900, 106901, 106873]","[112329, 112325, 112326]","[[-6.07698, 5.96967]]","[[-6.30051, 6.00826]]",[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

In [97]:
# write hexes to parquet
hexes.to_parquet(os.path.join(output_folder_path, "all_countries_hex8_complete_no_roads.parquet"), index=False)

### 4. to destination add bridge ids from bridge table hexes used columns for destination type
also, to bridge table add destination coords from destination table

In [105]:
bridges = gpd.read_parquet(bridges_path)
bridges

,bridge_index,type,geometry,subregion_indices,used_by_h3_for_semi_dense_urban_optimal,used_by_h3_for_health_posts_optimal,used_by_h3_for_primary_schools_fixed,used_by_h3_for_all_health_facilities_optimal,used_by_h3_for_health_centers_optimal,used_by_h3_for_major_hospitals_optimal,used_by_h3_for_major_roads_optimal,used_by_h3_for_secondary_schools_fixed,used_by_h3_for_all_education_facilities_fixed
0,302351,bridge_predicted,POINT (37.46229 4.84980),"[269620, 269629]","[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...","[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...",[],[886a4b39a7fffff],"[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...",[],[],[],[]
1,302352,bridge_predicted,POINT (37.46595 4.85105),"[269620, 270228]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a7fffff, 886a4b39a9f...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],[],[],[]
2,302353,bridge_predicted,POINT (37.46508 4.85778),"[269906, 270228]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff]","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],[],[],[]
3,302354,bridge_predicted,POINT (37.45727 4.86828),"[267568, 269906]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a9fffff, 886a4b39e7fffff]","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39abf...",[],[]
4,302355,bridge_predicted,POINT (37.45522 4.87006),"[267568, 269717]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b3937fffff, 886a4b39a9fffff, 886a4b39e5f...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39abf...",[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...
116269,105971,bridge_predicted,POINT (29.63156 -8.49877),"[149739, 150797]","[8896a9a205fffff, 8896a9a229fffff, 8896a9a241f...","[8896a9a205fffff, 8896a9a229fffff, 8896a9a241f...",[8896a9bad7fffff],"[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...","[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...","[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...",[],"[88961a7985fffff, 88961a79a9fffff, 88961a79adf...",[8896a9bad7fffff]
116270,105972,bridge_predicted,POINT (29.63656 -8.48563),"[149147, 150797]","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...",[],"[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f..."
116271,105973,bridge_predicted,POINT (29.11544 -8.47639),"[135473, 136007]",[8896f4d2d3fffff],[8896f4d2d3fffff],"[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...",[],[],"[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f..."
116272,105974,bridge_predicted,POINT (29.75404 -8.41852),"[153714, 154514]","[8896a98cb3fffff, 8896a98cbbfffff, 8896a9b90df...","[8896a9b909fffff, 8896a9b90bfffff, 8896a9b91df...","[8896a9b909fffff, 8896a9b90bfffff, 8896a9b91df...","[8896a9b909fffff, 8896a9b90bfffff, 8896a9b91df...","[8896a9b90dfffff, 8896a9b947fffff, 8896a9b96bf...","[8896a98cb3fffff, 8896a98cbbfffff, 8896a9b90df...",[],"[8896a98cb3fffff, 8896a98cbbfffff, 8896a9b90df...","[8896a9b909fffff, 8896a9b90bfffff, 8896a9b91df..."


In [120]:
bridges['health_destinations'] = [[] for _ in range(len(bridges))]

# Create a lookup dictionary: h3_value -> list of (bridge_index, bridge_row_idx)
h3_to_bridges = {}
for bridge_idx, bridge_row in bridges.iterrows():
    bridge_h3s = bridge_row['used_by_h3_for_all_health_facilities_optimal']
    bridge_index = bridge_row['bridge_index']
    
    for h3 in bridge_h3s:
        if h3 not in h3_to_bridges:
            h3_to_bridges[h3] = []
        h3_to_bridges[h3].append((bridge_index, bridge_idx))

# Now process destinations much faster
def find_matching_bridges_fast(h3_indices_array, destination_coords):
    matching_bridges = []
    matched_bridge_indices = set()  # Avoid duplicates
    
    for h3 in h3_indices_array:
        if h3 in h3_to_bridges:
            for bridge_index, bridge_row_idx in h3_to_bridges[h3]:
                if bridge_row_idx not in matched_bridge_indices:
                    matching_bridges.append(bridge_index)
                    bridges.at[bridge_row_idx, 'health_destinations'].append(destination_coords)
                    matched_bridge_indices.add(bridge_row_idx)
    
    return matching_bridges

health_destinations['bridges_used'] = health_destinations.apply(
    lambda row: find_matching_bridges_fast(row['h3_indices'], row['coords']), 
    axis=1
)

In [119]:
bridges['edu_destinations'] = [[] for _ in range(len(bridges))]

# Create a lookup dictionary: h3_value -> list of (bridge_index, bridge_row_idx)
h3_to_bridges = {}
for bridge_idx, bridge_row in bridges.iterrows():
    bridge_h3s = bridge_row['used_by_h3_for_all_education_facilities_fixed']
    bridge_index = bridge_row['bridge_index']
    
    for h3 in bridge_h3s:
        if h3 not in h3_to_bridges:
            h3_to_bridges[h3] = []
        h3_to_bridges[h3].append((bridge_index, bridge_idx))

# Now process destinations much faster
def find_matching_bridges_fast(h3_indices_array, destination_coords):
    matching_bridges = []
    matched_bridge_indices = set()  # Avoid duplicates
    
    for h3 in h3_indices_array:
        if h3 in h3_to_bridges:
            for bridge_index, bridge_row_idx in h3_to_bridges[h3]:
                if bridge_row_idx not in matched_bridge_indices:
                    matching_bridges.append(bridge_index)
                    bridges.at[bridge_row_idx, 'edu_destinations'].append(destination_coords)
                    matched_bridge_indices.add(bridge_row_idx)
    
    return matching_bridges

edu_destinations['bridges_used'] = edu_destinations.apply(
    lambda row: find_matching_bridges_fast(row['h3_indices'], row['coords']), 
    axis=1
)

In [118]:
bridges['market_destinations'] = [[] for _ in range(len(bridges))]

# Create a lookup dictionary: h3_value -> list of (bridge_index, bridge_row_idx)
h3_to_bridges = {}
for bridge_idx, bridge_row in bridges.iterrows():
    bridge_h3s = bridge_row['used_by_h3_for_semi_dense_urban_optimal']
    bridge_index = bridge_row['bridge_index']
    
    for h3 in bridge_h3s:
        if h3 not in h3_to_bridges:
            h3_to_bridges[h3] = []
        h3_to_bridges[h3].append((bridge_index, bridge_idx))

# Now process destinations much faster
def find_matching_bridges_fast(h3_indices_array, destination_coords):
    matching_bridges = []
    matched_bridge_indices = set()  # Avoid duplicates
    
    for h3 in h3_indices_array:
        if h3 in h3_to_bridges:
            for bridge_index, bridge_row_idx in h3_to_bridges[h3]:
                if bridge_row_idx not in matched_bridge_indices:
                    matching_bridges.append(bridge_index)
                    bridges.at[bridge_row_idx, 'market_destinations'].append(destination_coords)
                    matched_bridge_indices.add(bridge_row_idx)
    
    return matching_bridges

market_destinations['bridges_used'] = market_destinations.apply(
    lambda row: find_matching_bridges_fast(row['h3_indices'], row['coords']), 
    axis=1
)

In [121]:
bridges

,bridge_index,type,geometry,subregion_indices,used_by_h3_for_semi_dense_urban_optimal,used_by_h3_for_health_posts_optimal,used_by_h3_for_primary_schools_fixed,used_by_h3_for_all_health_facilities_optimal,used_by_h3_for_health_centers_optimal,used_by_h3_for_major_hospitals_optimal,used_by_h3_for_major_roads_optimal,used_by_h3_for_secondary_schools_fixed,used_by_h3_for_all_education_facilities_fixed,health_destinations,edu_destinations,market_destinations
0,302351,bridge_predicted,POINT (37.46229 4.84980),"[269620, 269629]","[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...","[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...",[],[886a4b39a7fffff],"[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...",[],[],[],[],"[[37.4615, 4.75478]]",[],[]
1,302352,bridge_predicted,POINT (37.46595 4.85105),"[269620, 270228]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a7fffff, 886a4b39a9f...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],[],[],[],"[[37.3238, 5.02786], [37.4615, 4.75478]]",[],[]
2,302353,bridge_predicted,POINT (37.46508 4.85778),"[269906, 270228]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff]","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],[],[],[],"[[37.3238, 5.02786], [37.4615, 4.75478]]",[],[]
3,302354,bridge_predicted,POINT (37.45727 4.86828),"[267568, 269906]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a9fffff, 886a4b39e7fffff]","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39abf...",[],[],"[[37.3238, 5.02786], [37.4615, 4.75478]]",[],[]
4,302355,bridge_predicted,POINT (37.45522 4.87006),"[267568, 269717]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b3937fffff, 886a4b39a9fffff, 886a4b39e5f...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39abf...",[],[],"[[37.3238, 5.02786], [37.4615, 4.75478]]",[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116269,105971,bridge_predicted,POINT (29.63156 -8.49877),"[149739, 150797]","[8896a9a205fffff, 8896a9a229fffff, 8896a9a241f...","[8896a9a205fffff, 8896a9a229fffff, 8896a9a241f...",[8896a9bad7fffff],"[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...","[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...","[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...",[],"[88961a7985fffff, 88961a79a9fffff, 88961a79adf...",[8896a9bad7fffff],"[[29.37935, -8.4282], [29.6395, -8.4262], [29....","[[29.62989, -8.52485], [29.6597, -8.48236]]",[]
116270,105972,bridge_predicted,POINT (29.63656 -8.48563),"[149147, 150797]","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...",[],"[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[[29.6395, -8.4262], [29.66422, -8.4704], [29....","[[29.59214, -8.4134], [29.62989, -8.52485], [2...",[]
116271,105973,bridge_predicted,POINT (29.11544 -8.47639),"[135473, 136007]",[8896f4d2d3fffff],[8896f4d2d3fffff],"[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...",[],[],"[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[[29.0869, -8.56953]]","[[29.09681, -8.4511], [29.08776, -8.51007], [2...",[]
116272,105974,bridge_predicted,POINT (29.75404 -8.41852),"[153714, 154514]","[8896a98cb3fffff, 8896a98cbbfffff, 8896a9b90df...","[8896a9b909fffff, 8896a9b90bfffff,

In [117]:
market_destinations

,category_value,geometry,category,coords,urban_center_index,subregion_index,h3_indices,bridges_used
0,30,POINT (42.05792 4.17458),urban_center,"[42.05792, 4.17458]",1,719554,"[887ae02001fffff, 887ae02003fffff, 887ae02005f...","[322351, 322353, 322354, 322355, 322352, 32235..."
1,30,POINT (42.17458 5.22458),urban_center,"[42.17458, 5.22458]",2,726112,"[887ac41001fffff, 887ac41003fffff, 887ac41009f...","[322774, 322777, 322779, 322791, 322842, 32284..."
2,30,POINT (41.89125 5.34125),urban_center,"[41.89125, 5.34125]",3,710027,"[887ac40083fffff, 887ac40091fffff, 887ac40093f...","[323091, 322828, 322829, 322836, 322838, 32283..."
3,30,POINT (43.55792 5.94125),urban_center,"[43.55792, 5.94125]",4,774776,"[887af02103fffff, 887af02107fffff, 887af02131f...","[323231, 323050, 323151, 323152, 323153, 32315..."
4,30,POINT (43.54125 5.94125),urban_center,"[43.54125, 5.94125]",5,774776,"[887af14183fffff, 887af14185fffff, 887af14187f...","[323248, 323249, 323244, 323245, 323250, 32436..."
...,...,...,...,...,...,...,...,...
3498,30,POINT (31.10792 -8.78292),urban_center,"[31.10792, -8.78292]",409,12829,[],[]
3499,30,POINT (31.12458 -8.78292),urban_center,"[31.12458, -8.78292]",410,204118,[],[]
3500,30,POINT (31.10792 -8.76625),urban_center,"[31.10792, -8.76625]",411,12847,[],[]
3501,30,POINT (31.12458 -8.76625),urban_center,"[31.12458, -8.76625]",412,203857,[],[]


In [126]:
# write bridges to parquet
bridges.to_parquet(os.path.join(output_folder_path, "all_bridges_full_no_roads.parquet"), index=False)

In [127]:
health_destinations.to_parquet(os.path.join(output_folder_path, "all_health_facilities_full_no_roads.parquet"), index=False)
edu_destinations.to_parquet(os.path.join(output_folder_path, "all_education_facilities_full_no_roads.parquet"), index=False)
market_destinations.to_parquet(os.path.join(output_folder_path, "all_urban_centers_full_no_roads.parquet"), index=False)

### 5. to bridge table add destination coords from destination table
did this in the above section

## Summary of outputs

In [129]:
hexes

,h3_index,population,pop_0_4,females_0_4,males_0_4,pop_5_9,females_5_9,males_5_9,pop_10_14,females_10_14,males_10_14,pop_0_9,females_0_9,males_0_9,pop_15_49,females_15_49,males_15_49,pop_50_64,females_50_64,males_50_64,pop_65_plus,females_65_plus,males_65_plus,births,pregnancies,rwi,underweight,female_educational_attainment_mean,male_educational_attainment_mean,travel_time_no_sites_all_health,time_delta_no_sites_semi_dense_urban,travel_time_health_posts,travel_time_major_roads,travel_time_no_sites_secondary_schools,travel_time_secondary_schools,travel_time_no_sites_health_centers,travel_time_no_sites_major_roads,time_delta_no_sites_secondary_schools,time_delta_no_sites_all_health,travel_time_health_centers,time_delta_no_sites_health_centers,time_delta_no_sites_major_roads,travel_time_semi_dense_urban,time_delta_no_sites_major_hospitals,travel_time_all_health,travel_time_no_sites_primary_schools,travel_time_no_sites_semi_dense_urban,time_delta_no_sites_health_posts,travel_time_no_sites_all_education,travel_time_major_hospitals,travel_time_no_sites_major_hospitals,travel_time_primary_schools,time_delta_no_sites_primary_schools,travel_time_all_education,time_delta_no_sites_all_education,travel_time_no_sites_health_posts,geometry,country_name,bridges_used_for_semi_dense_urban_optimal,bridges_used_for_health_posts_optimal,bridges_used_for_all_health_facilities_optimal,bridges_used_for_health_centers_optimal,bridges_used_for_major_hospitals_optimal,bridges_used_for_major_roads_optimal,bridges_used_for_primary_schools_fixed,bridges_used_for_secondary_schools_fixed,bridges_used_for_all_education_facilities_fixed,edu_destinations,health_destinations,market_destinations
0,887512209bfffff,5,1,0,0,0,0,0,0,0,0,1,0,0,2,1,1,0,0,0,0,0,0,0,0,-0.277,0.165,3.0,5.1,174.0,0.0,358,59,0.0,0,174.0,59.0,0.0,0.0,174,0.0,0.0,144,0.0,174,NaN,144.0,0.0,NaN,0,0.0,1607,NaN,1599,NaN,358.0,"POLYGON ((-6.50082 7.36543, -6.50476 7.36330, ...",civ,[],[107568],[],[],[],[],[107343],[],[107343],"[[-6.44833, 6.91472]]","[[-6.47955, 7.375]]","[[-6.49125, 7.37792]]"
1,8875ae4635fffff,22,3,2,1,3,1,1,2,1,1,7,3,3,10,4,5,1,0,0,0,0,0,0,0,-0.768,0.156,2.4,4.8,858.0,0.0,341,857,0.0,0,858.0,857.0,0.0,0.0,858,0.0,0.0,880,0.0,858,952.0,880.0,0.0,884.0,0,0.0,952,0.0,884,0.0,341.0,"POLYGON ((-8.16296 6.42187, -8.16688 6.41973, ...",civ,[107775],[],[],[],[],[107778],[106649],[],[107775],"[[-8.00614, 6.57902]]","[[-8.11667, 6.44413]]",[]
2,88753244dbfffff,40,7,3,3,6,3,3,4,2,2,13,6,6,17,9,8,3,1,1,1,0,0,0,0,-0.603,0.144,1.9,3.4,739.0,0.0,271,376,430.0,430,739.0,376.0,0.0,0.0,739,0.0,0.0,399,0.0,739,367.0,399.0,0.0,367.0,1154,1154.0,367,0.0,367,0.0,271.0,"POLYGON ((-5.39022 9.68681, -5.39424 9.68467, ...",civ,"[113616, 113620, 113613, 113621]","[113617, 113620, 113621]","[113617, 113620, 113621]",[113607],"[113620, 113621]","[113610, 113613, 113616, 113620, 113621]","[113610, 113613, 113616, 113620, 113621]","[113616, 113620, 113613, 113621]","[113610, 113613, 113616, 113620, 113621]","[[-5.37773, 9.58976]]","[[-5.435, 9.633]]",[]
3,8875ab8c3bfffff,8,1,0,0,1,0,0,0,0,0,2,1,1,4,2,2,0,0,0,0,0,0,0,0,-0.400,0.143,3.3,4.9,0.0,0.0,181,64,0.0,0,0.0,64.0,0.0,0.0,0,0.0,0.0,1047,0.0,0,0.0,1047.0,0.0,1426.0,897,897.0,0,0.0,1426,0.0,181.0,"POLYGON ((-7.33689 5.25591, -7.34074 5.25381, ...",civ,"[106087, 106088, 106096, 106098, 106077, 106079]",[],[],[],"[105947, 105948, 105941, 105943]",[],[],[],"[106087, 106088, 106096, 106098, 106077, 106079]","[[-7.36109, 5.62703]]","[[-7.31698, 5.27485]]",[]
4,8875ad3897fffff,25,4,2,2,3,1,1,3,1,1,8,3,4,11,5,5,1,0,0,0,0,0,1,2,-0.139,0.160,2.8,4.6,590.0,0.0,272,24,853.0,853,590.0,24.0,0.0,0.0,590,0.0,0.0,250,0.0,590,956.0,250.0,0.0,559.0,998,998.0,956,0.0,559,0.0,272.0,"POLYGON ((-6.22794 6.03084, -6.23181 6.02875, ...",civ,[],[],[],[],[],[],[],"[106897, 106898, 106900, 106901, 106873]","[112329, 112325, 112326]","[[-6.07698, 5.96967]]","[[-6.30051, 6.00826]]",[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,

In [130]:
bridges

,bridge_index,type,geometry,subregion_indices,used_by_h3_for_semi_dense_urban_optimal,used_by_h3_for_health_posts_optimal,used_by_h3_for_primary_schools_fixed,used_by_h3_for_all_health_facilities_optimal,used_by_h3_for_health_centers_optimal,used_by_h3_for_major_hospitals_optimal,used_by_h3_for_major_roads_optimal,used_by_h3_for_secondary_schools_fixed,used_by_h3_for_all_education_facilities_fixed,health_destinations,edu_destinations,market_destinations
0,302351,bridge_predicted,POINT (37.46229 4.84980),"[269620, 269629]","[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...","[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...",[],[886a4b39a7fffff],"[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...",[],[],[],[],"[[37.4615, 4.75478]]",[],[]
1,302352,bridge_predicted,POINT (37.46595 4.85105),"[269620, 270228]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a7fffff, 886a4b39a9f...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],[],[],[],"[[37.3238, 5.02786], [37.4615, 4.75478]]",[],[]
2,302353,bridge_predicted,POINT (37.46508 4.85778),"[269906, 270228]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff]","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],[],[],[],"[[37.3238, 5.02786], [37.4615, 4.75478]]",[],[]
3,302354,bridge_predicted,POINT (37.45727 4.86828),"[267568, 269906]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a9fffff, 886a4b39e7fffff]","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39abf...",[],[],"[[37.3238, 5.02786], [37.4615, 4.75478]]",[],[]
4,302355,bridge_predicted,POINT (37.45522 4.87006),"[267568, 269717]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b3937fffff, 886a4b39a9fffff, 886a4b39e5f...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39abf...",[],[],"[[37.3238, 5.02786], [37.4615, 4.75478]]",[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116269,105971,bridge_predicted,POINT (29.63156 -8.49877),"[149739, 150797]","[8896a9a205fffff, 8896a9a229fffff, 8896a9a241f...","[8896a9a205fffff, 8896a9a229fffff, 8896a9a241f...",[8896a9bad7fffff],"[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...","[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...","[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...",[],"[88961a7985fffff, 88961a79a9fffff, 88961a79adf...",[8896a9bad7fffff],"[[29.37935, -8.4282], [29.6395, -8.4262], [29....","[[29.62989, -8.52485], [29.6597, -8.48236]]",[]
116270,105972,bridge_predicted,POINT (29.63656 -8.48563),"[149147, 150797]","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...",[],"[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[[29.6395, -8.4262], [29.66422, -8.4704], [29....","[[29.59214, -8.4134], [29.62989, -8.52485], [2...",[]
116271,105973,bridge_predicted,POINT (29.11544 -8.47639),"[135473, 136007]",[8896f4d2d3fffff],[8896f4d2d3fffff],"[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...",[],[],"[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[[29.0869, -8.56953]]","[[29.09681, -8.4511], [29.08776, -8.51007], [2...",[]
116272,105974,bridge_predicted,POINT (29.75404 -8.41852),"[153714, 154514]","[8896a98cb3fffff, 8896a98cbbfffff, 8896a9b90df...","[8896a9b909fffff, 8896a9b90bfffff,

In [131]:
health_destinations

,facility_type,geometry,coords,all_health_facilities_index,subregion_index,name,fid,globalid,h3_indices,bridges_used
0,clinic,POINT (38.71860 9.02970),"[38.7186, 9.0297]",1,402987,Addis Ketema Clinic 1,NaN,None,[],[]
1,clinic,POINT (38.73860 9.02922),"[38.7386, 9.02922]",2,404694,Addis Ketema Clinic 10,NaN,None,[],[]
2,clinic,POINT (38.73620 9.02868),"[38.7362, 9.02868]",3,403784,Addis Ketema Clinic 11,NaN,None,[],[]
3,clinic,POINT (38.73470 9.02816),"[38.7347, 9.02816]",4,403784,Addis Ketema Clinic 12,NaN,None,[],[]
4,clinic,POINT (38.73400 9.02942),"[38.734, 9.02942]",5,403784,Addis Ketema Clinic 13,NaN,None,[],[]
...,...,...,...,...,...,...,...,...,...,...
26642,rural health post,POINT (23.58512 -16.68015),"[23.58512, -16.68015]",2601,27133,Sankandi RHP,2604.0,{F2987FF1-FEB8-437F-BE54-386D551B385C},"[8897582027fffff, 8897582113fffff, 889758211bf...","[93195, 93194, 93201, 93244, 93245, 93246, 932..."
26643,rural health center,POINT (23.15679 -16.19463),"[23.15679, -16.19463]",2602,21382,Silowana RHC,2605.0,{B2A502DC-2FD2-4BB5-9564-489FC20BFA8D},"[88975b1613fffff, 88975b1659fffff, 88975b165bf...","[93651, 93653, 93782, 93649, 93592, 93593, 936..."
26644,rural health center,POINT (23.02583 -17.31588),"[23.02583, -17.31588]",2603,19064,Sinjembela RHC,2606.0,{C80AE0FC-E7A1-4E5A-8AF2-BD191C211B4B},"[889758a42dfffff, 889758a445fffff, 889758a44bf...","[93174, 93177, 93204]"
26645,rural health center,POINT (23.50325 -16.60110),"[23.50325, -16.6011]",2604,25914,Sioma RHC,2607.0,{B61042BB-4AC6-4D0E-AC37-C75DBC8CFD92},"[8897582883fffff, 8897582889fffff, 889758288bf...","[93201, 93202, 93222]"


In [132]:
edu_destinations

,id,name,geometry,category,coords,all_education_facilities_index,subregion_index,fid,globalid,h3_indices,bridges_used
0,4527811.0,unity university,POINT (38.80434 9.00017),other,"[38.80434, 9.00017]",1,412081,NaN,None,"[88529b7901fffff, 88529b790dfffff]",[]
1,4527812.0,cpu college,POINT (38.80249 9.00068),college,"[38.80249, 9.00068]",2,412081,NaN,None,[],[]
2,4527813.0,ትምህርት ቤት,POINT (38.52361 9.77877),,"[38.52361, 9.77877]",3,378792,NaN,None,"[8852980907fffff, 8852980915fffff, 8852980917f...","[319629, 319697, 319863, 319870, 319935, 31994..."
3,4527814.0,walya primary,POINT (37.44922 12.59572),primary,"[37.44922, 12.59572]",4,269325,NaN,None,"[885282d011fffff, 885282d017fffff, 885282d019f...","[335595, 335594, 335915, 335916, 335917, 33591..."
4,4527815.0,further training institute university adama,POINT (39.28836 8.56323),other,"[39.28836, 8.56323]",5,466166,NaN,None,"[887aca0003fffff, 887aca0007fffff, 887aca0011f...","[317733, 317734, 317735, 317841]"
...,...,...,...,...,...,...,...,...,...,...,...
104781,NaN,Chikuse Primary School,POINT (28.67089 -14.50118),Primary School,"[28.67089, -14.50118]",7128,123849,8489.0,35LPD800961_002,"[889605c649fffff, 889605c64dfffff, 889605d485f...","[96339, 96340, 96341, 96350, 96351, 96353, 96338]"
104782,NaN,Chimpempe Secondary School,POINT (29.43418 -9.54560),Secondary School,"[29.43418, -9.5456]",7129,144698,8490.0,35LQK672438_001,"[88961a9833fffff, 88961a9837fffff, 88961a98e5f...",[]
104783,NaN,Kafubu Basic School,POINT (28.58740 -13.02607),Basic School,"[28.5874, -13.02607]",7130,120543,8502.0,35LPF721594_001,"[8896002031fffff, 8896002033fffff, 8896002035f...",[]
104784,NaN,Namiyanga School,POINT (26.50011 -17.07299),School,"[26.50011, -17.07299]",7131,68657,8506.0,35KMB468123_001,"[8897530b01fffff, 8897530b03fffff, 8897530b09f...","[91738, 91734, 91735]"


In [133]:
market_destinations

,category_value,geometry,category,coords,urban_center_index,subregion_index,h3_indices,bridges_used
0,30,POINT (42.05792 4.17458),urban_center,"[42.05792, 4.17458]",1,719554,"[887ae02001fffff, 887ae02003fffff, 887ae02005f...","[322351, 322353, 322354, 322355, 322352, 32235..."
1,30,POINT (42.17458 5.22458),urban_center,"[42.17458, 5.22458]",2,726112,"[887ac41001fffff, 887ac41003fffff, 887ac41009f...","[322774, 322777, 322779, 322791, 322842, 32284..."
2,30,POINT (41.89125 5.34125),urban_center,"[41.89125, 5.34125]",3,710027,"[887ac40083fffff, 887ac40091fffff, 887ac40093f...","[323091, 322828, 322829, 322836, 322838, 32283..."
3,30,POINT (43.55792 5.94125),urban_center,"[43.55792, 5.94125]",4,774776,"[887af02103fffff, 887af02107fffff, 887af02131f...","[323231, 323050, 323151, 323152, 323153, 32315..."
4,30,POINT (43.54125 5.94125),urban_center,"[43.54125, 5.94125]",5,774776,"[887af14183fffff, 887af14185fffff, 887af14187f...","[323248, 323249, 323244, 323245, 323250, 32436..."
...,...,...,...,...,...,...,...,...
3498,30,POINT (31.10792 -8.78292),urban_center,"[31.10792, -8.78292]",409,12829,[],[]
3499,30,POINT (31.12458 -8.78292),urban_center,"[31.12458, -8.78292]",410,204118,[],[]
3500,30,POINT (31.10792 -8.76625),urban_center,"[31.10792, -8.76625]",411,12847,[],[]
3501,30,POINT (31.12458 -8.76625),urban_center,"[31.12458, -8.76625]",412,203857,[],[]


## Switch Coords to ID's

In [3]:
bridges = gpd.read_parquet(os.path.join(output_folder_path, "all_bridges_full_no_roads.parquet"))
hexes = gpd.read_parquet(os.path.join(output_folder_path, "all_countries_hex8_complete_no_roads.parquet"))
health_destinations = gpd.read_parquet(os.path.join(output_folder_path, "all_health_facilities_full_no_roads.parquet"))
edu_destinations = gpd.read_parquet(os.path.join(output_folder_path, "all_education_facilities_full_no_roads.parquet"))
market_destinations = gpd.read_parquet(os.path.join(output_folder_path, "all_urban_centers_full_no_roads.parquet"))

In [4]:
def replace_coords_with_ids(edit_df, destinations_df, edit_col, index_col, coord_col='coords'):
    """Replace coordinates with IDs using exact string matching"""
    
    # Create lookup dictionary: coordinate string -> ID
    coord_to_id = dict(zip(destinations_df[coord_col].astype(str), destinations_df[index_col]))
    
    def replace_coord_list(coord_list):
        return [coord_to_id.get(str(coord), coord) for coord in coord_list]
    
    result_df = edit_df.copy()
    result_df[edit_col] = result_df[edit_col].apply(replace_coord_list)
    
    return result_df

In [5]:
bridges

,bridge_index,type,geometry,subregion_indices,used_by_h3_for_semi_dense_urban_optimal,used_by_h3_for_health_posts_optimal,used_by_h3_for_primary_schools_fixed,used_by_h3_for_all_health_facilities_optimal,used_by_h3_for_health_centers_optimal,used_by_h3_for_major_hospitals_optimal,used_by_h3_for_major_roads_optimal,used_by_h3_for_secondary_schools_fixed,used_by_h3_for_all_education_facilities_fixed,health_destinations,edu_destinations,market_destinations
0,302351,bridge_predicted,POINT (37.46229 4.84980),"[269620, 269629]","[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...","[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...",[],[886a4b39a7fffff],"[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...",[],[],[],[],"[[37.4615, 4.75478]]",[],[]
1,302352,bridge_predicted,POINT (37.46595 4.85105),"[269620, 270228]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a7fffff, 886a4b39a9f...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],[],[],[],"[[37.3238, 5.02786], [37.4615, 4.75478]]",[],[]
2,302353,bridge_predicted,POINT (37.46508 4.85778),"[269906, 270228]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff]","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],[],[],[],"[[37.3238, 5.02786], [37.4615, 4.75478]]",[],[]
3,302354,bridge_predicted,POINT (37.45727 4.86828),"[267568, 269906]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a9fffff, 886a4b39e7fffff]","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39abf...",[],[],"[[37.3238, 5.02786], [37.4615, 4.75478]]",[],[]
4,302355,bridge_predicted,POINT (37.45522 4.87006),"[267568, 269717]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b3937fffff, 886a4b39a9fffff, 886a4b39e5f...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39abf...",[],[],"[[37.3238, 5.02786], [37.4615, 4.75478]]",[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116269,105971,bridge_predicted,POINT (29.63156 -8.49877),"[149739, 150797]","[8896a9a205fffff, 8896a9a229fffff, 8896a9a241f...","[8896a9a205fffff, 8896a9a229fffff, 8896a9a241f...",[8896a9bad7fffff],"[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...","[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...","[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...",[],"[88961a7985fffff, 88961a79a9fffff, 88961a79adf...",[8896a9bad7fffff],"[[29.37935, -8.4282], [29.6395, -8.4262], [29....","[[29.62989, -8.52485], [29.6597, -8.48236]]",[]
116270,105972,bridge_predicted,POINT (29.63656 -8.48563),"[149147, 150797]","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...",[],"[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[[29.6395, -8.4262], [29.66422, -8.4704], [29....","[[29.59214, -8.4134], [29.62989, -8.52485], [2...",[]
116271,105973,bridge_predicted,POINT (29.11544 -8.47639),"[135473, 136007]",[8896f4d2d3fffff],[8896f4d2d3fffff],"[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...",[],[],"[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[[29.0869, -8.56953]]","[[29.09681, -8.4511], [29.08776, -8.51007], [2...",[]
116272,105974,bridge_predicted,POINT (29.75404 -8.41852),"[153714, 154514]","[8896a98cb3fffff, 8896a98cbbfffff, 8896a9b90df...","[8896a9b909fffff, 8896a9b90bfffff,

In [ ]:
bridges_updated = replace_coords_with_ids(bridges, health_destinations, 'health_destinations', 'all_health_facilities_index')


,bridge_index,type,geometry,subregion_indices,used_by_h3_for_semi_dense_urban_optimal,used_by_h3_for_health_posts_optimal,used_by_h3_for_primary_schools_fixed,used_by_h3_for_all_health_facilities_optimal,used_by_h3_for_health_centers_optimal,used_by_h3_for_major_hospitals_optimal,used_by_h3_for_major_roads_optimal,used_by_h3_for_secondary_schools_fixed,used_by_h3_for_all_education_facilities_fixed,health_destinations,edu_destinations,market_destinations
0,302351,bridge_predicted,POINT (37.46229 4.84980),"[269620, 269629]","[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...","[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...",[],[886a4b39a7fffff],"[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...",[],[],[],[],[2904],[],[]
1,302352,bridge_predicted,POINT (37.46595 4.85105),"[269620, 270228]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a7fffff, 886a4b39a9f...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],[],[],[],"[2212, 2904]",[],[]
2,302353,bridge_predicted,POINT (37.46508 4.85778),"[269906, 270228]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff]","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],[],[],[],"[2212, 2904]",[],[]
3,302354,bridge_predicted,POINT (37.45727 4.86828),"[267568, 269906]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a9fffff, 886a4b39e7fffff]","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39abf...",[],[],"[2212, 2904]",[],[]
4,302355,bridge_predicted,POINT (37.45522 4.87006),"[267568, 269717]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b3937fffff, 886a4b39a9fffff, 886a4b39e5f...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39abf...",[],[],"[2212, 2904]",[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116269,105971,bridge_predicted,POINT (29.63156 -8.49877),"[149739, 150797]","[8896a9a205fffff, 8896a9a229fffff, 8896a9a241f...","[8896a9a205fffff, 8896a9a229fffff, 8896a9a241f...",[8896a9bad7fffff],"[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...","[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...","[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...",[],"[88961a7985fffff, 88961a79a9fffff, 88961a79adf...",[8896a9bad7fffff],"[1034, 1035, 1586, 1587]","[[29.62989, -8.52485], [29.6597, -8.48236]]",[]
116270,105972,bridge_predicted,POINT (29.63656 -8.48563),"[149147, 150797]","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...",[],"[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[1035, 1586, 1587]","[[29.59214, -8.4134], [29.62989, -8.52485], [2...",[]
116271,105973,bridge_predicted,POINT (29.11544 -8.47639),"[135473, 136007]",[8896f4d2d3fffff],[8896f4d2d3fffff],"[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...",[],[],"[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...",[1038],"[[29.09681, -8.4511], [29.08776, -8.51007], [2...",[]
116272,105974,bridge_predicted,POINT (29.75404 -8.41852),"[153714, 154514]","[8896a98cb3fffff, 8896a98cbbfffff, 8896a9b90df...","[8896a9b909fffff, 8896a9b90bfffff, 8896a9b91df...","[8896a9b909fffff, 8896a9b90bfffff, 8896a9b91df...","[8896a9b909fffff, 8896a9b90bfffff, 8896a9b91df...","[8896a9b90dfffff, 8896a9b947fffff, 8896a9b96bf...","[8896a98cb3fffff, 8896a98cb

In [7]:
bridges_updated = replace_coords_with_ids(bridges_updated, edu_destinations, 'edu_destinations', 'all_education_facilities_index')
bridges_updated = replace_coords_with_ids(bridges_updated, market_destinations, 'market_destinations', 'urban_center_index')
bridges_updated

,bridge_index,type,geometry,subregion_indices,used_by_h3_for_semi_dense_urban_optimal,used_by_h3_for_health_posts_optimal,used_by_h3_for_primary_schools_fixed,used_by_h3_for_all_health_facilities_optimal,used_by_h3_for_health_centers_optimal,used_by_h3_for_major_hospitals_optimal,used_by_h3_for_major_roads_optimal,used_by_h3_for_secondary_schools_fixed,used_by_h3_for_all_education_facilities_fixed,health_destinations,edu_destinations,market_destinations
0,302351,bridge_predicted,POINT (37.46229 4.84980),"[269620, 269629]","[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...","[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...",[],[886a4b39a7fffff],"[886a4b2b0dfffff, 886a4b2b41fffff, 886a4b2b45f...",[],[],[],[],[2904],[],[]
1,302352,bridge_predicted,POINT (37.46595 4.85105),"[269620, 270228]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a7fffff, 886a4b39a9f...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],[],[],[],"[2212, 2904]",[],[]
2,302353,bridge_predicted,POINT (37.46508 4.85778),"[269906, 270228]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff]","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],[],[],[],"[2212, 2904]",[],[]
3,302354,bridge_predicted,POINT (37.45727 4.86828),"[267568, 269906]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a9fffff, 886a4b39e7fffff]","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39abf...",[],[],"[2212, 2904]",[],[]
4,302355,bridge_predicted,POINT (37.45522 4.87006),"[267568, 269717]","[886a4b2941fffff, 886a4b294bfffff, 886a4b294df...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b3937fffff, 886a4b39a9fffff, 886a4b39e5f...","[886a4b2b0dfffff, 886a4b2b21fffff, 886a4b2b25f...",[],"[886a4b39a1fffff, 886a4b39a9fffff, 886a4b39abf...",[],[],"[2212, 2904]",[],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
116269,105971,bridge_predicted,POINT (29.63156 -8.49877),"[149739, 150797]","[8896a9a205fffff, 8896a9a229fffff, 8896a9a241f...","[8896a9a205fffff, 8896a9a229fffff, 8896a9a241f...",[8896a9bad7fffff],"[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...","[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...","[8896a9a201fffff, 8896a9a205fffff, 8896a9a209f...",[],"[88961a7985fffff, 88961a79a9fffff, 88961a79adf...",[8896a9bad7fffff],"[1034, 1035, 1586, 1587]","[6112, 6115]",[]
116270,105972,bridge_predicted,POINT (29.63656 -8.48563),"[149147, 150797]","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...",[],"[8896a9a001fffff, 8896a9a009fffff, 8896a9a00df...","[8896a9a301fffff, 8896a9a303fffff, 8896a9a305f...","[1035, 1586, 1587]","[1377, 6112, 6115]",[]
116271,105973,bridge_predicted,POINT (29.11544 -8.47639),"[135473, 136007]",[8896f4d2d3fffff],[8896f4d2d3fffff],"[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...","[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...",[],[],"[88961a6da5fffff, 8896f4d281fffff, 8896f4d283f...",[1038],"[1387, 1388, 1389]",[]
116272,105974,bridge_predicted,POINT (29.75404 -8.41852),"[153714, 154514]","[8896a98cb3fffff, 8896a98cbbfffff, 8896a9b90df...","[8896a9b909fffff, 8896a9b90bfffff, 8896a9b91df...","[8896a9b909fffff, 8896a9b90bfffff, 8896a9b91df...","[8896a9b909fffff, 8896a9b90bfffff, 8896a9b91df...","[8896a9b90dfffff, 8896a9b947fffff, 8896a9b96bf...","[8896a98cb3fffff, 8896a98cbbfffff, 8896a9b90df...",[],"[8896a98cb3fffff, 8896a98cbbfffff, 8896a9b90df...","[8896a9b909ff

In [8]:
hexes_updated = replace_coords_with_ids(hexes, health_destinations, 'health_destinations', 'all_health_facilities_index')
hexes_updated = replace_coords_with_ids(hexes_updated, edu_destinations, 'edu_destinations', 'all_education_facilities_index')
hexes_updated = replace_coords_with_ids(hexes_updated, market_destinations, 'market_destinations', 'urban_center_index')
hexes_updated

,h3_index,population,pop_0_4,females_0_4,males_0_4,pop_5_9,females_5_9,males_5_9,pop_10_14,females_10_14,males_10_14,pop_0_9,females_0_9,males_0_9,pop_15_49,females_15_49,males_15_49,pop_50_64,females_50_64,males_50_64,pop_65_plus,females_65_plus,males_65_plus,births,pregnancies,rwi,underweight,female_educational_attainment_mean,male_educational_attainment_mean,travel_time_no_sites_all_health,time_delta_no_sites_semi_dense_urban,travel_time_health_posts,travel_time_major_roads,travel_time_no_sites_secondary_schools,travel_time_secondary_schools,travel_time_no_sites_health_centers,travel_time_no_sites_major_roads,time_delta_no_sites_secondary_schools,time_delta_no_sites_all_health,travel_time_health_centers,time_delta_no_sites_health_centers,time_delta_no_sites_major_roads,travel_time_semi_dense_urban,time_delta_no_sites_major_hospitals,travel_time_all_health,travel_time_no_sites_primary_schools,travel_time_no_sites_semi_dense_urban,time_delta_no_sites_health_posts,travel_time_no_sites_all_education,travel_time_major_hospitals,travel_time_no_sites_major_hospitals,travel_time_primary_schools,time_delta_no_sites_primary_schools,travel_time_all_education,time_delta_no_sites_all_education,travel_time_no_sites_health_posts,geometry,country_name,bridges_used_for_semi_dense_urban_optimal,bridges_used_for_health_posts_optimal,bridges_used_for_all_health_facilities_optimal,bridges_used_for_health_centers_optimal,bridges_used_for_major_hospitals_optimal,bridges_used_for_major_roads_optimal,bridges_used_for_primary_schools_fixed,bridges_used_for_secondary_schools_fixed,bridges_used_for_all_education_facilities_fixed,edu_destinations,health_destinations,market_destinations
0,887512209bfffff,5,1,0,0,0,0,0,0,0,0,1,0,0,2,1,1,0,0,0,0,0,0,0,0,-0.277,0.165,3.0,5.1,174.0,0.0,358,59,0.0,0,174.0,59.0,0.0,0.0,174,0.0,0.0,144,0.0,174,NaN,144.0,0.0,NaN,0,0.0,1607,NaN,1599,NaN,358.0,"POLYGON ((-6.50082 7.36543, -6.50476 7.36330, ...",civ,[],[107568],[],[],[],[],[107343],[],[107343],[1231],[1252],[99]
1,8875ae4635fffff,22,3,2,1,3,1,1,2,1,1,7,3,3,10,4,5,1,0,0,0,0,0,0,0,-0.768,0.156,2.4,4.8,858.0,0.0,341,857,0.0,0,858.0,857.0,0.0,0.0,858,0.0,0.0,880,0.0,858,952.0,880.0,0.0,884.0,0,0.0,952,0.0,884,0.0,341.0,"POLYGON ((-8.16296 6.42187, -8.16688 6.41973, ...",civ,[107775],[],[],[],[],[107778],[106649],[],[107775],[211],[971],[]
2,88753244dbfffff,40,7,3,3,6,3,3,4,2,2,13,6,6,17,9,8,3,1,1,1,0,0,0,0,-0.603,0.144,1.9,3.4,739.0,0.0,271,376,430.0,430,739.0,376.0,0.0,0.0,739,0.0,0.0,399,0.0,739,367.0,399.0,0.0,367.0,1154,1154.0,367,0.0,367,0.0,271.0,"POLYGON ((-5.39022 9.68681, -5.39424 9.68467, ...",civ,"[113616, 113620, 113613, 113621]","[113617, 113620, 113621]","[113617, 113620, 113621]",[113607],"[113620, 113621]","[113610, 113613, 113616, 113620, 113621]","[113610, 113613, 113616, 113620, 113621]","[113616, 113620, 113613, 113621]","[113610, 113613, 113616, 113620, 113621]",[1397],[1385],[]
3,8875ab8c3bfffff,8,1,0,0,1,0,0,0,0,0,2,1,1,4,2,2,0,0,0,0,0,0,0,0,-0.400,0.143,3.3,4.9,0.0,0.0,181,64,0.0,0,0.0,64.0,0.0,0.0,0,0.0,0.0,1047,0.0,0,0.0,1047.0,0.0,1426.0,897,897.0,0,0.0,1426,0.0,181.0,"POLYGON ((-7.33689 5.25591, -7.34074 5.25381, ...",civ,"[106087, 106088, 106096, 106098, 106077, 106079]",[],[],[],"[105947, 105948, 105941, 105943]",[],[],[],"[106087, 106088, 106096, 106098, 106077, 106079]",[2389],[238],[]
4,8875ad3897fffff,25,4,2,2,3,1,1,3,1,1,8,3,4,11,5,5,1,0,0,0,0,0,1,2,-0.139,0.160,2.8,4.6,590.0,0.0,272,24,853.0,853,590.0,24.0,0.0,0.0,590,0.0,0.0,250,0.0,590,956.0,250.0,0.0,559.0,998,998.0,956,0.0,559,0.0,272.0,"POLYGON ((-6.22794 6.03084, -6.23181 6.02875, ...",civ,[],[],[],[],[],[],[],"[106897, 106898, 106900, 106901, 106873]","[112329, 112325, 112326]",[1952],[479],[]
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
206

In [9]:
# overwrite all output files
bridges_updated.to_parquet(os.path.join(output_folder_path, "all_bridges_full_no_roads.parquet"), index=False)
hexes_updated.to_parquet(os.path.join(output_folder_path, "all_countries_hex8_complete_no_roads.parquet"), index=False)
health_destinations.to_parquet(os.path.join(output_folder_path, "all_health_facilities_full_no_roads.parquet"), index=False)
edu_destinations.to_parquet(os.path.join(output_folder_path, "all_education_facilities_full_no_roads.parquet"), index=False)
market_destinations.to_parquet(os.path.join(output_folder_path, "all_urban_centers_full_no_roads.parquet"), index=False)

## Database Upload
```
ogr2ogr -f "PostgreSQL" PG:"host=impact-map.c45m2ca2ulxg.us-east-1.rds.amazonaws.com dbname=impact-map user=username password=password" /Volumes/samsung-4tb/b2p/impact-model/database_files/all_bridges_full_no_roads.parquet -nln bridges

ogr2ogr -f "PostgreSQL" PG:"host=impact-map.c45m2ca2ulxg.us-east-1.rds.amazonaws.com dbname=impact-map user=username password=password" /Volumes/samsung-4tb/b2p/impact-model/database_files/all_countries_hex8_complete_no_roads.parquet -nln hex_areas

ogr2ogr -f "PostgreSQL" PG:"host=impact-map.c45m2ca2ulxg.us-east-1.rds.amazonaws.com dbname=impact-map user=username password=password" /Volumes/samsung-4tb/b2p/impact-model/database_files/all_health_facilities_full_no_roads.parquet -nln health_facilities

ogr2ogr -f "PostgreSQL" PG:"host=impact-map.c45m2ca2ulxg.us-east-1.rds.amazonaws.com dbname=impact-map user=username password=password" /Volumes/samsung-4tb/b2p/impact-model/database_files/all_education_facilities_full_no_roads.parquet -nln education_facilities

ogr2ogr -f "PostgreSQL" PG:"host=impact-map.c45m2ca2ulxg.us-east-1.rds.amazonaws.com dbname=impact-map user=username password=password" /Volumes/samsung-4tb/b2p/impact-model/database_files/all_urban_centers_full_no_roads.parquet -nln urban_centers
```
Create an index:
`CREATE INDEX idx_column_name ON table_name (column_name);`

CREATE INDEX idx_h3_index_all_paths ON all_paths (h3_index);
CREATE INDEX idx_h3_index_hex_areas ON hex_areas (h3_index);
CREATE INDEX idx_bridge_index_bridges ON bridges (bridge_index);
CREATE INDEX idx_urban_center_index_urban_centers ON urban_centers (urban_center_index);
CREATE INDEX idx_health_facilities_index_health_facilities ON health_facilities (all_health_facilities_index);
CREATE INDEX idx_education_facilities_index_education_facilities ON education_facilities (all_education_facilities_index);


urban_center_index
all_health_facilities_index
all_education_facilities_index